# CNN'lerde Nümerik Optimizasyon Algoritmalarının Yakınsama ve Kararlılık Analizi

**MAT353 – Nümerik Analiz · 2026–2027 Güz · İstanbul Medeniyet Üniversitesi**

**Ekip:** _Ad Soyad (Öğrenci No)_, _Ad Soyad (Öğrenci No)_

SGD, Momentum, RMSprop ve Adam optimizasyon algoritmaları formüllerinden sıfırdan yazılmış (`optimizers.py`) ve sığ (3 konvolüsyon) ile BatchNorm'suz derin (8 konvolüsyon) CNN'lerde Fashion-MNIST üzerinde karşılaştırılmıştır.

| Deney | Soru | Çıktı |
|---|---|---|
| 1 | Yakınsama hızı ve seed kararlılığı (5 seed, ortalama ± std) | `exp1_summary.csv`, `loss_convergence.png`, `gradient_norm.png`, `time_vs_acc.png` |
| 2 | Öğrenme oranı duyarlılığı ve nümerik patlama (NaN/Inf) | `exp2_lr_stability.csv`, `lr_heatmap.png` |
| 3 | Gradyan doğrulaması: sonlu fark kesme/yuvarlama hatası | `exp3_fd_error.csv`, `fd_error.png` |
| 4 | CPU – GPU performans karşılaştırması | `exp4_cpu_gpu.csv`, `cpu_gpu.png` |
| 5 | Hessian'ın en büyük özdeğeri (kuvvet yöntemi) ve teorik kararlılık sınırı η < 2/λ_max | `exp5_sharpness.csv`, `sharpness.png` |
| 6 | Ablasyon: BatchNorm ve Adam yanlılık düzeltmesi (eşleştirilmiş t-testi) | `exp6_summary.csv`, `ablation.png` |

**Tekrarlanabilirlik:** seed'ler 42, 100, 2024, 7, 99; `random`, `numpy`, `torch` seed'leri ve `cudnn.deterministic=True` sabitlenmiştir.

> Çalıştırma: *Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU*, ardından *Tümünü çalıştır*.

In [ ]:
# === Ayarlar ===
RUN_FULL = False   # True: TÜM deneyleri baştan çalıştırır (T4 GPU'da ~1.5 saat).
                   # False: yalnızca sonucu eksik olan deneyler çalışır, kayıtlı sonuçlar kullanılır.
USE_DRIVE = True   # Sonuçları Google Drive'da kalıcı tut (oturum kopsa da CSV'ler kaybolmaz)

In [ ]:
# === Çalışma klasörü ===
import os

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJ = '/content/drive/MyDrive/MAT353'
else:
    PROJ = '/content/MAT353'
os.makedirs(PROJ, exist_ok=True)
%cd {PROJ}

## 0. Proje dosyaları
Aşağıdaki hücreler projenin Python modüllerini çalışma klasörüne yazar. Kodu incelemek için hücreleri açabilirsiniz.

In [ ]:
%%writefile models.py
"""
models.py
=========
Deneylerde kullanılan iki CNN mimarisi.

* ShallowCNN : 3 konvolüsyon katmanı (sığ, kararlı referans model)
* DeepCNN    : 8 konvolüsyon katmanı, BatchNorm YOK.
               Normalizasyon katmanı bilerek kullanılmamıştır; böylece derinlik
               arttıkça geri yayılımda gradyanların katmandan katmana çarpılarak
               sönmesi (vanishing) veya patlaması (exploding) ve buna bağlı
               nümerik kararsızlıklar (NaN / Inf) gözlemlenebilir.

Girdi: 1x28x28 gri seviye görüntü (FashionMNIST), çıktı: 10 sınıf logit'i.
"""

import torch.nn as nn
import torch.nn.functional as F


class ShallowCNN(nn.Module):
    """3 konvolüsyon + 2 tam bağlantılı katmandan oluşan sığ CNN."""

    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 28x28 -> 14x14
        x = self.pool(F.relu(self.conv2(x)))  # 14x14 -> 7x7
        x = F.relu(self.conv3(x))             # 7x7
        x = x.flatten(1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


class DeepCNN(nn.Module):
    """
    8 konvolüsyon katmanlı, BatchNorm KULLANMAYAN derin CNN.

    Her konvolüsyon 3x3, padding=1. 2. ve 4. konvolüsyondan sonra MaxPool
    uygulanır (28 -> 14 -> 7). Ağırlıklar PyTorch'un varsayılan başlatmasıyla
    (kaiming_uniform, a=sqrt(5)) başlatılır; bu başlatma ReLU için varyansı
    korumadığından derin ağda aktivasyon ve gradyan ölçekleri katman katman
    küçülür -> gradyan sönmesi gözlemlenir. Yüksek öğrenme oranlarında ise
    ağırlık güncellemeleri büyüyüp patlamaya (NaN/Inf) yol açabilir.
    """

    # Kanal sayıları: girdi + 8 konvolüsyonun çıkışları
    CHANNELS = [1, 16, 16, 32, 32, 32, 32, 64, 64]
    POOL_AFTER = {1, 3}  # 0-indeksli: conv2 ve conv4'ten sonra havuzlama

    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.convs = nn.ModuleList(
            nn.Conv2d(cin, cout, kernel_size=3, padding=1)
            for cin, cout in zip(self.CHANNELS[:-1], self.CHANNELS[1:])
        )
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(self.CHANNELS[-1] * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        for i, conv in enumerate(self.convs):
            x = F.relu(conv(x))
            if i in self.POOL_AFTER:
                x = self.pool(x)
        x = x.flatten(1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


class DeepCNN_BN(DeepCNN):
    """
    ABLASYON MODELİ: DeepCNN ile birebir aynı mimari, tek fark her konvolüsyondan
    sonra BatchNorm2d. BatchNorm her katmanın çıktısını mini-batch üzerinde
    sıfır ortalama / birim varyansa çeker:  x̂ = (x − μ_B) / √(σ²_B + ε),  y = γx̂ + β.
    Böylece katmanlar arası ölçek küçülmesi (dolayısıyla gradyan sönmesi) engellenir.
    DeepCNN'de SGD'nin takılmasının nedeni gerçekten sönme ise bu modelde SGD öğrenmelidir.
    """

    def __init__(self, num_classes: int = 10):
        super().__init__(num_classes)
        self.bns = nn.ModuleList(nn.BatchNorm2d(c) for c in self.CHANNELS[1:])

    def forward(self, x):
        for i, (conv, bn) in enumerate(zip(self.convs, self.bns)):
            x = F.relu(bn(conv(x)))
            if i in self.POOL_AFTER:
                x = self.pool(x)
        x = x.flatten(1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


MODEL_REGISTRY = {"ShallowCNN": ShallowCNN, "DeepCNN": DeepCNN, "DeepCNN_BN": DeepCNN_BN}


def build_model(name: str) -> nn.Module:
    """İsme göre model örneği oluşturur."""
    if name not in MODEL_REGISTRY:
        raise ValueError(f"Bilinmeyen model: {name}. Seçenekler: {list(MODEL_REGISTRY)}")
    return MODEL_REGISTRY[name]()


def weight_layers(model: nn.Module):
    """
    Ağırlık taşıyan katmanları (Conv2d / Linear) girdiden çıktıya doğru sırayla
    döndürür. Her eleman: (okunabilir_ad, modül). Örn. ('conv1', Conv2d(...)).
    Gradyan normu ölçümlerinde 'ilk katman' = listenin başı, 'son katman' = sonu.
    """
    layers, n_conv, n_fc = [], 0, 0
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            n_conv += 1
            layers.append((f"conv{n_conv}", module))
        elif isinstance(module, nn.Linear):
            n_fc += 1
            layers.append((f"fc{n_fc}", module))
    return layers


if __name__ == "__main__":
    # Hızlı kontrol: parametre sayıları ve katman listesi
    import torch

    for name in MODEL_REGISTRY:
        m = build_model(name)
        out = m(torch.zeros(2, 1, 28, 28))
        n_params = sum(p.numel() for p in m.parameters())
        print(f"{name}: çıktı {tuple(out.shape)}, {n_params:,} parametre, "
              f"katmanlar = {[n for n, _ in weight_layers(m)]}")


In [ ]:
%%writefile optimizers.py
"""
optimizers.py
=============
SGD, Momentum, RMSprop ve Adam optimizasyon algoritmalarının formüllerinden
SIFIRDAN yazılmış uygulamaları. Her sınıf torch.optim.Optimizer arayüzünü
kullanır; böylece eğitim döngüsü değişmeden kullanılabilir.

Gösterim:  θ_t  parametre vektörü,  g_t = ∇_θ L(θ_{t-1})  mini-batch gradyanı,
           η  öğrenme oranı,  ⊙  eleman bazlı çarpım,  t = 1, 2, ...

    SGD        θ_t = θ_{t-1} − η g_t

    Momentum   v_t = μ v_{t-1} + g_t                       (v_0 = 0, μ = 0.9)
               θ_t = θ_{t-1} − η v_t

    RMSprop    s_t = α s_{t-1} + (1 − α) g_t ⊙ g_t         (s_0 = 0, α = 0.99)
               θ_t = θ_{t-1} − η g_t / (√s_t + ε)

    Adam       m_t = β1 m_{t-1} + (1 − β1) g_t             (β1 = 0.9)
               v_t = β2 v_{t-1} + (1 − β2) g_t ⊙ g_t       (β2 = 0.999)
               m̂_t = m_t / (1 − β1^t),   v̂_t = v_t / (1 − β2^t)   (yanlılık düzeltmesi)
               θ_t = θ_{t-1} − η m̂_t / (√v̂_t + ε)

Hesap karmaşıklığı: P parametre sayısı olmak üzere her adım O(P) zaman alır.
Ek bellek: SGD O(1), Momentum O(P), RMSprop O(P), Adam O(2P).

İşlem sırası PyTorch'un referans uygulamasıyla aynı tutulmuştur; tests.py
içindeki birim testleri iki uygulamanın kayan nokta hassasiyetinde
eşleştiğini doğrular.
"""

import math

import torch
from torch.optim import Optimizer


class ManualSGD(Optimizer):
    """Saf SGD ve (momentum > 0 ise) ağır top (heavy-ball) momentumlu SGD."""

    def __init__(self, params, lr=0.01, momentum=0.0):
        if lr <= 0 or momentum < 0:
            raise ValueError("lr > 0 ve momentum >= 0 olmalı")
        super().__init__(params, dict(lr=lr, momentum=momentum))

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            lr, mu = group["lr"], group["momentum"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad
                if mu > 0:
                    state = self.state[p]
                    if "v" not in state:
                        state["v"] = g.clone()            # v_1 = g_1  (v_0 = 0)
                    else:
                        state["v"].mul_(mu).add_(g)       # v_t = μ v_{t-1} + g_t
                    g = state["v"]
                p.add_(g, alpha=-lr)                      # θ_t = θ_{t-1} − η (·)
        return None


class ManualRMSprop(Optimizer):
    """RMSprop: gradyan karelerinin üstel hareketli ortalaması ile ölçekleme."""

    def __init__(self, params, lr=0.001, alpha=0.99, eps=1e-8):
        if lr <= 0 or not 0 <= alpha < 1:
            raise ValueError("lr > 0 ve 0 <= alpha < 1 olmalı")
        super().__init__(params, dict(lr=lr, alpha=alpha, eps=eps))

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            lr, a, eps = group["lr"], group["alpha"], group["eps"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad
                state = self.state[p]
                if "s" not in state:
                    state["s"] = torch.zeros_like(p)
                s = state["s"]
                s.mul_(a).addcmul_(g, g, value=1 - a)     # s_t = α s + (1−α) g²
                denom = s.sqrt().add_(eps)                # √s_t + ε
                p.addcdiv_(g, denom, value=-lr)           # θ −= η g / denom
        return None


class ManualAdam(Optimizer):
    """
    Adam: birinci ve ikinci moment tahminleri + yanlılık düzeltmesi.

    bias_correction=False (ABLASYON): m̂ = m, v̂ = v alınır. m_0 = v_0 = 0 başlangıcı
    nedeniyle E[m_t] = (1 − β1^t) E[g] ve E[v_t] = (1 − β2^t) E[g²] olduğundan, düzeltme
    yapılmazsa ilk adımın büyüklüğü η(1 − β1)/√(1 − β2) ≈ 3.16 η olur (β1=0.9, β2=0.999);
    yani ilk iterasyonlarda etkin öğrenme oranı ~3 kat büyür.
    """

    def __init__(self, params, lr=0.001, betas=(0.9, 0.999), eps=1e-8, bias_correction=True):
        if lr <= 0 or not (0 <= betas[0] < 1 and 0 <= betas[1] < 1):
            raise ValueError("lr > 0 ve 0 <= beta < 1 olmalı")
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps,
                                      bias_correction=bias_correction))

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            lr, (b1, b2), eps = group["lr"], group["betas"], group["eps"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad
                state = self.state[p]
                if not state:
                    state["t"] = 0
                    state["m"] = torch.zeros_like(p)
                    state["v"] = torch.zeros_like(p)
                state["t"] += 1
                t, m, v = state["t"], state["m"], state["v"]

                m.lerp_(g, 1 - b1)                        # m_t = β1 m + (1−β1) g
                v.mul_(b2).addcmul_(g, g, value=1 - b2)   # v_t = β2 v + (1−β2) g²

                if group["bias_correction"]:
                    bc1 = 1 - b1 ** t                     # yanlılık düzeltme katsayıları
                    bc2 = 1 - b2 ** t
                else:
                    bc1 = bc2 = 1.0                       # ablasyon: düzeltme yok
                # η m̂/(√v̂ + ε) = (η/bc1) · m / (√v/√bc2 + ε)
                denom = (v.sqrt() / math.sqrt(bc2)).add_(eps)
                p.addcdiv_(m, denom, value=-lr / bc1)
        return None


def make_optimizer(name: str, params, lr: float, impl: str = "manual") -> Optimizer:
    """
    name : 'SGD' | 'Momentum' | 'RMSprop' | 'Adam'
    impl : 'manual' -> bu dosyadaki sıfırdan uygulamalar
           'torch'  -> PyTorch referans uygulamaları (doğrulama için)
    """
    if impl == "manual":
        table = {
            "SGD": lambda: ManualSGD(params, lr=lr),
            "Momentum": lambda: ManualSGD(params, lr=lr, momentum=0.9),
            "RMSprop": lambda: ManualRMSprop(params, lr=lr),
            "Adam": lambda: ManualAdam(params, lr=lr),
            "Adam-noBC": lambda: ManualAdam(params, lr=lr, bias_correction=False),
        }
    elif impl == "torch":
        table = {
            "SGD": lambda: torch.optim.SGD(params, lr=lr),
            "Momentum": lambda: torch.optim.SGD(params, lr=lr, momentum=0.9),
            "RMSprop": lambda: torch.optim.RMSprop(params, lr=lr, alpha=0.99, eps=1e-8),
            "Adam": lambda: torch.optim.Adam(params, lr=lr, betas=(0.9, 0.999), eps=1e-8),
        }
    else:
        raise ValueError(f"Bilinmeyen uygulama: {impl}")
    if name not in table:
        raise ValueError(f"Bilinmeyen optimizer: {name}")
    return table[name]()


In [ ]:
%%writefile gradient_check.py
"""
gradient_check.py
=================
Geri yayılımla (otomatik türev) hesaplanan gradyanın merkezi sonlu farkla
doğrulanması ve sonlu fark yaklaşımının HATA ANALİZİ.

Kayıp fonksiyonu L: R^P -> R, birim yön vektörü v (||v||_2 = 1) için yönlü türev

    D_v L(θ) = ∇L(θ) · v                                  (otomatik türev, "kesin")

ve merkezi sonlu fark yaklaşımı

    D_h = [ L(θ + h v) − L(θ − h v) ] / (2h).

Taylor açılımından toplam hata iki bileşenlidir:

    |D_h − D_v L|  ≈  (h² / 6) |L'''|      +   ε_mach |L| / h
                      kesme hatası O(h²)        yuvarlama hatası O(ε/h)

İkisini dengeleyen en iyi adım h* ≈ (3 ε_mach |L| / |L'''|)^(1/3) ~ ε_mach^(1/3):
    float64 (ε ≈ 2.2e-16) -> h* ~ 1e-5      float32 (ε ≈ 1.2e-7) -> h* ~ 1e-2..1e-3
Grafikte log-log ölçekte sağda eğimi +2, solda eğimi −1 olan "V" şekli beklenir.

Not: ReLU ve MaxPool kırık (türevi süreksiz) noktalar içerir; büyük h bu
noktaları aşarsa kesme hatası O(h²)'den daha hızlı büyüyebilir.
"""

import copy

import numpy as np
import torch
import torch.nn.functional as F

from models import build_model


def _set_params(params, base, direction, t):
    """θ ← θ0 + t·v  (yerinde, gradyan izlenmeden)."""
    with torch.no_grad():
        for p, p0, v in zip(params, base, direction):
            p.copy_(p0 + t * v)


def directional_fd_errors(model_name, X, y, hs, seed=42, batch=32):
    """
    Bir model için farklı h değerlerinde merkezi sonlu farkın göreli hatasını
    hem float64 hem float32 aritmetiğinde hesaplar.

    Referans (kesin) değer her zaman float64 otomatik türevden alınır.
    Dönen liste: [{model, dtype, h, fd, exact, rel_error}, ...]
    """
    torch.manual_seed(seed)
    model64 = build_model(model_name).double().cpu()
    xb64 = X[:batch].detach().cpu().double()
    yb = y[:batch].detach().cpu()

    # --- Kesin yönlü türev (float64, otomatik türev) ---
    params64 = list(model64.parameters())
    loss = F.cross_entropy(model64(xb64), yb)
    grads = torch.autograd.grad(loss, params64)
    gen = torch.Generator().manual_seed(seed + 1)
    direction = [torch.randn(p.shape, generator=gen, dtype=torch.float64) for p in params64]
    vnorm = torch.sqrt(sum((d ** 2).sum() for d in direction))
    direction = [d / vnorm for d in direction]                      # ||v||_2 = 1
    exact = float(sum((g * d).sum() for g, d in zip(grads, direction)))

    rows = []
    for dtype in (torch.float64, torch.float32):
        model = copy.deepcopy(model64).to(dtype)
        params = list(model.parameters())
        base = [p.detach().clone() for p in params]
        dirs = [d.to(dtype) for d in direction]
        xb = xb64.to(dtype)
        model.eval()
        for h in hs:
            with torch.no_grad():
                _set_params(params, base, dirs, +h)
                l_plus = F.cross_entropy(model(xb), yb).item()
                _set_params(params, base, dirs, -h)
                l_minus = F.cross_entropy(model(xb), yb).item()
            _set_params(params, base, dirs, 0.0)
            fd = (l_plus - l_minus) / (2 * h)
            rows.append(dict(model=model_name, dtype=str(dtype).replace("torch.", ""), h=h,
                             fd=fd, exact=exact,
                             rel_error=abs(fd - exact) / max(abs(exact), 1e-30)))
    return rows


def default_hs():
    """10^-12 ... 10^0 arası logaritmik h ızgarası."""
    return list(np.logspace(-12, 0, 25))


In [ ]:
%%writefile hessian.py
"""
hessian.py
==========
Kayıp fonksiyonunun Hessian matrisinin en büyük özdeğeri λ_max'ın KUVVET YÖNTEMİ
(power iteration) ile kestirimi ve bundan öğrenme oranı için kararlılık sınırı.

1) Hessian–vektör çarpımı (Pearlmutter, 1994)
   H = ∇²L(θ) ∈ R^{P×P} açıkça oluşturulmaz (P ≈ 10^5 için P² ≈ 10^10 eleman).
   Bunun yerine Hv, gradyanın v yönündeki türevi olarak iki geri yayılımla hesaplanır:
       H v = ∇_θ ( ∇_θ L(θ) · v )
   Maliyet: yaklaşık 2 gradyan hesabı, bellek O(P).

2) Kuvvet yöntemi
       v_0 rastgele, ||v_0||_2 = 1
       w_k = H v_{k-1}
       λ_k = v_{k-1}ᵀ w_k            (Rayleigh bölümü)
       v_k = w_k / ||w_k||_2
   H simetrik ve |λ_1| > |λ_2| ise Rayleigh bölümünün hatası
       |λ_k − λ_1| = O( |λ_2 / λ_1|^{2k} ).
   Durdurma ölçütü: |λ_k − λ_{k-1}| ≤ tol · |λ_k|.
   Toplam maliyet: K iterasyon × O(N·P)  (N = örnek sayısı, P = parametre sayısı).

3) Kararlılık sınırı (yerel ikinci derece model L(θ) ≈ L* + ½ (θ−θ*)ᵀ H (θ−θ*))
   Gradyan inişi:  e_{k+1} = (I − ηH) e_k  →  özdeğer yönünde çarpan (1 − ηλ_i).
       |1 − ηλ_i| < 1  ∀i   ⇔   0 < η < 2 / λ_max
   Ağır top momentumu (μ):  0 < η < 2(1 + μ) / λ_max.
   Bu sınırların ötesinde en dik eğrilik yönünde hata her adımda büyür (ıraksama).
"""

import torch
import torch.nn.functional as F


def _dot(a, b):
    return sum((x * y).sum() for x, y in zip(a, b))


def hvp(loss_fn, params, vec):
    """Hessian–vektör çarpımı Hv (iki geri yayılım, Hessian oluşturulmaz)."""
    loss = loss_fn()
    grads = torch.autograd.grad(loss, params, create_graph=True)
    return torch.autograd.grad(_dot(grads, vec), params)


def top_eigenvalue(loss_fn, params, n_iter=100, tol=1e-3, seed=0):
    """
    Kuvvet yöntemiyle Hessian'ın en büyük (mutlak değerce) özdeğeri.

    loss_fn : parametre almayan, skaler kayıp döndüren fonksiyon (closure)
    Dönen  : (λ, [λ_1, λ_2, ...] iterasyon geçmişi)
    """
    params = [p for p in params if p.requires_grad]
    gen = torch.Generator().manual_seed(seed)
    v = [torch.randn(p.shape, generator=gen).to(device=p.device, dtype=p.dtype) for p in params]
    norm = torch.sqrt(_dot(v, v))
    v = [x / norm for x in v]

    history = []
    lam = 0.0
    for _ in range(n_iter):
        w = hvp(loss_fn, params, v)
        lam = float(_dot(v, w))                       # Rayleigh bölümü vᵀHv
        history.append(lam)
        norm = float(torch.sqrt(_dot(w, w)))
        if norm == 0.0:                               # H v = 0 (ör. ölü ağ)
            break
        v = [x.detach() / norm for x in w]
        if len(history) > 1 and abs(history[-1] - history[-2]) <= tol * abs(history[-1]):
            break
    return lam, history


def model_sharpness(model, X, y, n_iter=100, tol=1e-3, seed=0):
    """Çapraz entropi kaybının (X, y üzerinde) Hessian'ının λ_max'ı."""
    was_training = model.training
    model.eval()
    lam, hist = top_eigenvalue(lambda: F.cross_entropy(model(X), y),
                               list(model.parameters()), n_iter=n_iter, tol=tol, seed=seed)
    model.train(was_training)
    return lam, hist


def critical_lr(lam, momentum=0.0):
    """Yerel ikinci derece modele göre kararlılık sınırı: η_kritik = 2(1+μ)/λ_max."""
    return 2.0 * (1.0 + momentum) / lam if lam > 0 else float("inf")


In [ ]:
%%writefile experiments.py
"""
experiments.py
==============
CNN'lerde nümerik optimizasyon algoritmalarının (SGD, Momentum, RMSprop, Adam)
derinlik ve gradyan kararsızlığı koşullarında yakınsama/kararlılık analizi.

Deney 1 - Yakınsama ve seed kararlılığı
    ShallowCNN ve DeepCNN, 4 optimizer x 5 seed x 10 epoch.
    Kaydedilenler: epoch başına eğitim kaybı, test doğruluğu, süre (s) ve
    ilk/son katman L2 gradyan normları ||grad||_2 (+ tüm katmanlar).
    -> results/exp1_epochs.csv        (ham, epoch bazlı)
    -> results/exp1_layer_grads.csv   (katman bazlı gradyan normları)
    -> results/exp1_runs.csv          (koşu başına son değerler)
    -> results/exp1_summary.csv       (ortalama ± standart sapma)

Deney 2 - Öğrenme oranı hassasiyeti ve nümerik patlama
    DeepCNN, 4 optimizer x lr = [0.5, 0.1, 0.01, 0.001, 0.0001] x 3 seed.
    NaN / Inf oluşan (taşma/patlama) konfigürasyonlar tespit edilir.
    -> results/exp2_lr_stability.csv   (koşu bazlı)
    -> results/exp2_summary.csv        (seed'ler üzerinden özet)
    -> results/exp2_epochs.csv

Deney 3 - Gradyan doğrulaması ve sonlu fark hata analizi
    Otomatik türev gradyanı merkezi sonlu farkla karşılaştırılır; h'ye göre
    kesme (O(h²)) ve yuvarlama (O(ε/h)) hataları float32/float64 için ölçülür.
    -> results/exp3_fd_error.csv

Deney 4 - CPU / GPU performans karşılaştırması
    İterasyon başına süre (ileri + geri + optimizer adımı).
    -> results/exp4_cpu_gpu.csv

Deney 5 - Hessian spektrumu ve kararlılık sınırı (kuvvet yöntemi)
    λ_max = Hessian'ın en büyük özdeğeri, Hessian–vektör çarpımı + kuvvet yöntemiyle
    eğitim boyunca (epoch 0..E) ölçülür; η_kritik = 2/λ_max teorik sınırı Deney 2'deki
    deneysel kararlılık sınırıyla karşılaştırılır.
    -> results/exp5_sharpness.csv, results/exp5_power_history.csv

Deney 6 - Ablasyon
    (a) DeepCNN + BatchNorm: gradyan sönmesinin nedeninin normalizasyon eksikliği
        olduğunu sınar (SGD, Momentum).
    (b) Adam yanlılık düzeltmesi kapalı: düzeltmenin erken iterasyonlardaki etkisi.
    5 seed, Deney 1 ile aynı seed'ler -> eşleştirilmiş (paired) t-testi.
    -> results/exp6_epochs.csv, results/exp6_runs.csv, results/exp6_summary.csv

Optimizer'lar optimizers.py içinde formüllerinden sıfırdan yazılmıştır
(--optim-impl manual, varsayılan). --optim-impl torch ile PyTorch'un referans
uygulamaları kullanılır; ikisinin eşdeğerliği tests.py ile doğrulanır.

Kullanım:
    python experiments.py                 # tüm deneyler + grafikler
    python experiments.py --exp 1         # yalnızca Deney 1
    python experiments.py --quick         # küçük veri alt kümesiyle hızlı duman testi
    python experiments.py --resummarize   # mevcut CSV'lerden exp1_summary.csv'yi yeniden üret
"""

import os

# CUDA üzerinde deterministik cuBLAS için torch import edilmeden ÖNCE ayarlanmalı
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import argparse
import math
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy import stats
from torchvision import datasets

from gradient_check import default_hs, directional_fd_errors
from hessian import critical_lr, model_sharpness
from models import build_model, weight_layers
from optimizers import make_optimizer

# ---------------------------------------------------------------------------
# Sabitler
# ---------------------------------------------------------------------------
SEEDS = [42, 100, 2024, 7, 99]
MODELS = ["ShallowCNN", "DeepCNN"]
OPTIMIZERS = ["SGD", "Momentum", "RMSprop", "Adam"]
LR_GRID = [0.5, 0.1, 0.01, 0.001, 0.0001]

# Deney 1'de her optimizer için literatürde yaygın varsayılan öğrenme oranları
DEFAULT_LR = {"SGD": 0.01, "Momentum": 0.01, "RMSprop": 0.001, "Adam": 0.001}

# FashionMNIST eğitim kümesi ortalama / standart sapması (normalizasyon için)
FMNIST_MEAN, FMNIST_STD = 0.2860, 0.3530

# Şans düzeyi 10 sınıf için %10; altında kalan sonlu koşular "takıldı" sayılır
STUCK_ACC_THRESHOLD = 0.15

# Başlangıç kaybı ≈ ln(10) ≈ 2.30. Sonlu ama bunun 10 katını aşan eğitim kaybı
# "patladı" sayılır (taşma eşiğine ulaşmamış nümerik ıraksama).
EXPLODED_LOSS_THRESHOLD = 10 * math.log(10)

# Yakınsama hızı ölçütü: test doğruluğunun ilk kez bu eşiğe ulaştığı epoch/iterasyon
TARGET_ACC = 0.85


# ---------------------------------------------------------------------------
# Tekrarlanabilirlik
# ---------------------------------------------------------------------------
def set_seed(seed: int) -> None:
    """Tüm rastgelelik kaynaklarını sabitler (deterministik sonuçlar için)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Deterministik karşılığı olmayan bir işlem varsa hata yerine uyarı ver
    torch.use_deterministic_algorithms(True, warn_only=True)


def classify_status(status, train_losses, final_acc):
    """
    Koşu durumunu belirler:
      nan_inf  : kayıp veya gradyan NaN/Inf oldu (kayan nokta taşması)
      exploded : sayılar sonlu kaldı ama eğitim kaybı EXPLODED_LOSS_THRESHOLD'u aştı
                 ve model şans düzeyinde kaldı (taşmaya varmamış ıraksama)
      stuck    : kayıp patlamadı ama doğruluk şans düzeyinde (< STUCK_ACC_THRESHOLD)
      stable   : öğrenme gerçekleşti
    """
    if status == "nan_inf":
        return status
    if not (final_acc >= STUCK_ACC_THRESHOLD):
        losses = np.asarray(train_losses, dtype=float)
        if np.any(np.isfinite(losses) & (losses > EXPLODED_LOSS_THRESHOLD)):
            return "exploded"
        return "stuck"
    return "stable"


def gpu_warmup(data, device, batch_size, n_iters=3):
    """
    İlk CUDA/cuDNN çağrısı (bağlam oluşturma, çekirdek seçimi) birkaç saniye sürer.
    Isınma yapılmazsa bu süre ilk koşunun ilk epoch'una eklenir ve süre ölçümlerini
    bozar. Her koşu set_seed() ile başladığından ısınma sonuçları etkilemez.
    """
    if device.type != "cuda":
        return
    xb, yb = data[0][:batch_size], data[1][:batch_size]
    for name in MODELS:
        model = build_model(name).to(device)
        opt = make_optimizer("Adam", model.parameters(), 1e-3)
        for _ in range(n_iters):
            opt.zero_grad(set_to_none=True)
            F.cross_entropy(model(xb), yb).backward()
            opt.step()
    sync(device)


def sync(device: torch.device) -> None:
    """GPU'da süre ölçümünün doğru olması için asenkron çekirdekleri bekler."""
    if device.type == "cuda":
        torch.cuda.synchronize()


# ---------------------------------------------------------------------------
# Veri
# ---------------------------------------------------------------------------
def load_fashion_mnist(device, data_dir="data", train_subset=None, test_subset=None):
    """
    FashionMNIST'i indirir ve tamamını normalize edilmiş tensörler olarak
    belleğe (gerekirse GPU'ya) yükler. DataLoader yerine bellek içi indeksleme
    kullanmak hem hızlıdır hem de karıştırma sırasını tam kontrol etmemizi sağlar.
    """
    train = datasets.FashionMNIST(data_dir, train=True, download=True)
    test = datasets.FashionMNIST(data_dir, train=False, download=True)

    def prep(ds, n):
        X = ds.data.float().div(255.0).unsqueeze(1)          # (N, 1, 28, 28)
        X = (X - FMNIST_MEAN) / FMNIST_STD
        y = ds.targets.clone().long()
        if n:
            X, y = X[:n], y[:n]
        return X.to(device), y.to(device)

    Xtr, ytr = prep(train, train_subset)
    Xte, yte = prep(test, test_subset)
    print(f"Veri yüklendi: eğitim {tuple(Xtr.shape)}, test {tuple(Xte.shape)}, cihaz={device}")
    return Xtr, ytr, Xte, yte


# ---------------------------------------------------------------------------
# Eğitim / değerlendirme
# ---------------------------------------------------------------------------
def train_one_epoch(model, optimizer, layers, X, y, batch_size, generator):
    """
    Bir epoch eğitir. Her mini-batch'te:
      * kayıp sonlu mu (NaN/Inf) kontrol edilir,
      * her ağırlık katmanının gradyanının L2 (Frobenius) normu ||grad||_2 ölçülür.
    Dönen sözlük: ortalama kayıp, katman bazlı ortalama gradyan normları,
    epoch içindeki en büyük toplam gradyan normu ve patlama bilgisi.
    """
    model.train()
    n = X.shape[0]
    perm = torch.randperm(n, generator=generator).to(X.device)

    loss_sum, n_batches = 0.0, 0
    grad_sums = np.zeros(len(layers), dtype=np.float64)
    max_total_grad = 0.0

    for start in range(0, n, batch_size):
        idx = perm[start:start + batch_size]
        xb, yb = X[idx], y[idx]

        optimizer.zero_grad(set_to_none=True)
        loss = F.cross_entropy(model(xb), yb)
        loss_val = loss.item()

        # İleri geçişte taşma: kayıp NaN veya Inf
        if not math.isfinite(loss_val):
            return dict(diverged=True, diverged_iter=n_batches + 1, train_loss=float("nan"),
                        grad_norms=np.full(len(layers), np.nan), max_grad=float("inf"))

        loss.backward()

        # Katman bazlı gradyan L2 normları (float64'e çevirip toplam normu hesapla)
        norms = torch.stack([m.weight.grad.detach().norm(2) for _, m in layers])
        norms = norms.double().cpu().numpy()
        total = float(np.sqrt(np.sum(norms ** 2)))

        # Geri yayılımda taşma: gradyan NaN veya Inf
        if not np.all(np.isfinite(norms)):
            return dict(diverged=True, diverged_iter=n_batches + 1, train_loss=float("nan"),
                        grad_norms=np.full(len(layers), np.nan), max_grad=float("inf"))

        optimizer.step()

        loss_sum += loss_val
        grad_sums += norms
        max_total_grad = max(max_total_grad, total)
        n_batches += 1

    return dict(diverged=False, diverged_iter=None, train_loss=loss_sum / n_batches,
                grad_norms=grad_sums / n_batches, max_grad=max_total_grad)


@torch.no_grad()
def evaluate(model, X, y, batch_size=1000):
    """Test doğruluğu ve ortalama test kaybını döndürür."""
    model.eval()
    n, correct, loss_sum = X.shape[0], 0, 0.0
    for s in range(0, n, batch_size):
        out = model(X[s:s + batch_size])
        loss_sum += F.cross_entropy(out, y[s:s + batch_size], reduction="sum").item()
        correct += (out.argmax(1) == y[s:s + batch_size]).sum().item()
    return correct / n, loss_sum / n


def run_training(model_name, opt_name, lr, seed, epochs, data, batch_size, device, tag="",
                 impl="manual"):
    """
    Tek bir (model, optimizer, lr, seed) koşusunu çalıştırır.
    NaN/Inf tespit edilirse eğitim o noktada durdurulur.
    """
    Xtr, ytr, Xte, yte = data
    iters_per_epoch = math.ceil(Xtr.shape[0] / batch_size)

    # Aynı seed -> aynı başlangıç ağırlıkları ve aynı mini-batch sırası.
    # Böylece optimizer'lar birebir aynı başlangıç koşullarında karşılaştırılır.
    set_seed(seed)
    model = build_model(model_name).to(device)
    layers = weight_layers(model)
    optimizer = make_optimizer(opt_name, model.parameters(), lr, impl=impl)
    gen = torch.Generator().manual_seed(seed)

    epoch_rows, layer_rows = [], []
    status, div_epoch, div_iter = "stable", None, None
    cum_time, max_grad = 0.0, 0.0
    base = dict(model=model_name, optimizer=opt_name, lr=lr, seed=seed)

    for epoch in range(1, epochs + 1):
        sync(device)
        t0 = time.perf_counter()
        r = train_one_epoch(model, optimizer, layers, Xtr, ytr, batch_size, gen)
        sync(device)
        dt = time.perf_counter() - t0
        cum_time += dt
        max_grad = max(max_grad, r["max_grad"])

        if r["diverged"]:
            status, div_epoch, div_iter = "nan_inf", epoch, r["diverged_iter"]
            epoch_rows.append(dict(base, epoch=epoch,
                                   iteration=(epoch - 1) * iters_per_epoch + div_iter,
                                   train_loss=np.nan, test_loss=np.nan,
                                   test_acc=np.nan, epoch_time=dt, cum_time=cum_time,
                                   grad_first=np.nan, grad_last=np.nan))
            print(f"  {tag} epoch {epoch:2d}: NaN/Inf tespit edildi (iterasyon {div_iter}) -> durduruldu")
            break

        test_acc, test_loss = evaluate(model, Xte, yte)
        # Son optimizer adımı ağırlıkları bozmuş olabilir (test kaybı sonlu değil)
        if not math.isfinite(test_loss):
            status, div_epoch = "nan_inf", epoch

        epoch_rows.append(dict(base, epoch=epoch, iteration=epoch * iters_per_epoch,
                               train_loss=r["train_loss"], test_loss=test_loss,
                               test_acc=test_acc, epoch_time=dt, cum_time=cum_time,
                               grad_first=r["grad_norms"][0], grad_last=r["grad_norms"][-1]))
        for li, ((lname, _), g) in enumerate(zip(layers, r["grad_norms"])):
            layer_rows.append(dict(base, epoch=epoch, layer_idx=li, layer_name=lname, grad_norm=g))

        print(f"  {tag} epoch {epoch:2d}: loss={r['train_loss']:.4f}  acc={test_acc * 100:5.2f}%  "
              f"|g1|={r['grad_norms'][0]:.2e}  |gL|={r['grad_norms'][-1]:.2e}  t={dt:.1f}s")

        if status == "nan_inf":
            print(f"  {tag} test kaybı sonlu değil -> durduruldu")
            epoch_rows[-1]["test_acc"] = np.nan
            break

    final = epoch_rows[-1]
    status = classify_status(status, [e["train_loss"] for e in epoch_rows], final["test_acc"])

    summary = dict(base, status=status, diverged_epoch=div_epoch, diverged_iter=div_iter,
                   epochs_completed=len(epoch_rows), final_train_loss=final["train_loss"],
                   final_test_acc=final["test_acc"],
                   best_test_acc=np.nanmax([e["test_acc"] for e in epoch_rows])
                   if any(np.isfinite(e["test_acc"]) for e in epoch_rows) else np.nan,
                   iters_per_epoch=iters_per_epoch,
                   total_time=cum_time, max_grad_norm=max_grad,
                   final_grad_first=final["grad_first"], final_grad_last=final["grad_last"])
    return epoch_rows, layer_rows, summary


# ---------------------------------------------------------------------------
# Yardımcılar
# ---------------------------------------------------------------------------
def save_csv(rows, path):
    pd.DataFrame(rows).to_csv(path, index=False)


def fmt_pm(values, fmt="{:.4f}"):
    """Bir dizi değeri 'ortalama ± std' metnine çevirir (örnek std, ddof=1)."""
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    if v.size == 0:
        return "NaN"
    std = v.std(ddof=1) if v.size > 1 else 0.0
    return f"{fmt.format(v.mean())} ± {fmt.format(std)}"


def ci95_halfwidth(values):
    """Student-t dağılımı ile %95 güven aralığı yarı genişliği."""
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    if v.size < 2:
        return np.nan
    return stats.t.ppf(0.975, v.size - 1) * v.std(ddof=1) / math.sqrt(v.size)


def epochs_to_target(epochs_df: pd.DataFrame, target=TARGET_ACC) -> pd.DataFrame:
    """Her koşu için test doğruluğunun ilk kez `target`'a ulaştığı epoch (yoksa NaN)."""
    out = []
    for (model, opt, seed), g in epochs_df.groupby(["model", "optimizer", "seed"]):
        hit = g[g.test_acc >= target].epoch
        out.append(dict(model=model, optimizer=opt, seed=seed,
                        epochs_to_target=hit.min() if len(hit) else np.nan))
    return pd.DataFrame(out)


def summarize_exp1(runs_df: pd.DataFrame, epochs_df: pd.DataFrame,
                   iters_per_epoch: int) -> pd.DataFrame:
    """Koşu bazlı sonuçları (model, optimizer) bazında ortalama ± std'ye indirger."""
    ett = epochs_to_target(epochs_df)
    rows = []
    for model in MODELS:
        for opt in OPTIMIZERS:
            g = runs_df[(runs_df.model == model) & (runs_df.optimizer == opt)]
            if g.empty:
                continue
            acc = g.final_test_acc * 100
            acc_f = acc[np.isfinite(acc)]
            e = ett[(ett.model == model) & (ett.optimizer == opt)].epochs_to_target
            n_hit = int(np.isfinite(e).sum())
            rows.append({
                "model": model,
                "optimizer": opt,
                "lr": g.lr.iloc[0],
                "n_seeds": len(g),
                "n_diverged": int(g.status.isin(["nan_inf", "exploded"]).sum()),
                f"reached_{int(TARGET_ACC * 100)}%": f"{n_hit}/{len(e)}",
                f"epochs_to_{int(TARGET_ACC * 100)}%": fmt_pm(e, "{:.1f}") if n_hit else "ulaşılamadı",
                f"iters_to_{int(TARGET_ACC * 100)}%": fmt_pm(e * iters_per_epoch, "{:.0f}")
                if n_hit else "ulaşılamadı",
                "final_train_loss": fmt_pm(g.final_train_loss),
                "final_test_acc(%)": fmt_pm(acc, "{:.2f}"),
                "test_acc_95CI(%)": f"± {ci95_halfwidth(acc):.2f}",
                "test_acc_CV(%)": f"{(acc_f.std(ddof=1) / acc_f.mean() * 100):.3f}"
                if len(acc_f) > 1 else "NaN",
                "best_test_acc(%)": fmt_pm(g.best_test_acc * 100, "{:.2f}"),
                "total_time(s)": fmt_pm(g.total_time, "{:.1f}"),
                "grad_norm_first_layer": fmt_pm(g.final_grad_first, "{:.3e}"),
                "grad_norm_last_layer": fmt_pm(g.final_grad_last, "{:.3e}"),
            })
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Deneyler
# ---------------------------------------------------------------------------
def experiment_1(args, data, device):
    print("\n" + "=" * 70)
    print("DENEY 1: Yakınsama ve seed kararlılığı")
    print("=" * 70)
    epoch_rows, layer_rows, run_rows = [], [], []
    total = len(args.models) * len(OPTIMIZERS) * len(args.seeds)
    k = 0
    for model_name in args.models:
        for opt in OPTIMIZERS:
            for seed in args.seeds:
                k += 1
                tag = f"[{k}/{total}] {model_name:<10} {opt:<8} seed={seed:<4}"
                print(tag)
                e, l, s = run_training(model_name, opt, DEFAULT_LR[opt], seed, args.epochs,
                                       data, args.batch_size, device, tag="   ",
                                       impl=args.optim_impl)
                epoch_rows += e
                layer_rows += l
                run_rows.append(s)
                # Her koşudan sonra kaydet: uzun deney yarıda kesilse de veri kaybolmaz
                save_csv(epoch_rows, os.path.join(args.results_dir, "exp1_epochs.csv"))
                save_csv(layer_rows, os.path.join(args.results_dir, "exp1_layer_grads.csv"))
                save_csv(run_rows, os.path.join(args.results_dir, "exp1_runs.csv"))

    write_exp1_summary(pd.DataFrame(run_rows), pd.DataFrame(epoch_rows),
                       math.ceil(data[0].shape[0] / args.batch_size), args.results_dir)


def write_exp1_summary(runs_df, epochs_df, iters_per_epoch, results_dir):
    summary = summarize_exp1(runs_df, epochs_df, iters_per_epoch)
    path = os.path.join(results_dir, "exp1_summary.csv")
    summary.to_csv(path, index=False, encoding="utf-8-sig")  # BOM: Excel'de '±' doğru görünsün
    print("\nDeney 1 özeti:")
    print(summary.to_string(index=False))
    print(f"-> {path}")


def experiment_2(args, data, device):
    print("\n" + "=" * 70)
    print("DENEY 2: Öğrenme oranı hassasiyeti ve nümerik patlama (DeepCNN)")
    print("=" * 70)
    seeds = args.seeds[:args.exp2_seeds]
    epoch_rows, rows = [], []
    total = len(OPTIMIZERS) * len(LR_GRID) * len(seeds)
    k = 0
    for opt in OPTIMIZERS:
        for lr in LR_GRID:
            for seed in seeds:
                k += 1
                tag = f"[{k}/{total}] DeepCNN {opt:<8} lr={lr:<7g} seed={seed:<4}"
                print(tag)
                e, _, s = run_training("DeepCNN", opt, lr, seed, args.exp2_epochs,
                                       data, args.batch_size, device, tag="   ",
                                       impl=args.optim_impl)
                epoch_rows += e
                rows.append(s)
                save_csv(epoch_rows, os.path.join(args.results_dir, "exp2_epochs.csv"))

    df = pd.DataFrame(rows)
    df["nan_or_inf"] = df.status == "nan_inf"
    cols = ["optimizer", "lr", "seed", "status", "nan_or_inf", "diverged_epoch", "diverged_iter",
            "epochs_completed", "final_train_loss", "final_test_acc", "best_test_acc",
            "max_grad_norm", "total_time"]
    df = df[cols]
    path = os.path.join(args.results_dir, "exp2_lr_stability.csv")
    df.to_csv(path, index=False)
    print(f"-> {path}")

    summary = summarize_exp2(df)
    path = os.path.join(args.results_dir, "exp2_summary.csv")
    summary.to_csv(path, index=False, encoding="utf-8-sig")
    print("\nDeney 2 özeti (seed'ler üzerinden):")
    print(summary.to_string(index=False))
    print(f"-> {path}")


STATUS_TR = {"stable": "kararlı", "stuck": "takıldı", "exploded": "patladı", "nan_inf": "NaN/Inf"}


def summarize_exp2(df: pd.DataFrame) -> pd.DataFrame:
    """Her (optimizer, lr) için durum sayıları ve son doğruluk ortalama ± std."""
    rows = []
    for opt in OPTIMIZERS:
        for lr in LR_GRID:
            g = df[(df.optimizer == opt) & (df.lr == lr)]
            if g.empty:
                continue
            counts = g.status.value_counts()
            rows.append({
                "optimizer": opt, "lr": lr, "n_seeds": len(g),
                "durum": ", ".join(f"{STATUS_TR.get(s, s)} {n}/{len(g)}"
                                   for s, n in counts.items()),
                "n_nan_inf": int(counts.get("nan_inf", 0)),
                "n_stable": int(counts.get("stable", 0)),
                "final_test_acc(%)": fmt_pm(g.final_test_acc * 100, "{:.2f}"),
                "max_grad_norm(max)": f"{g.max_grad_norm.max():.3g}",
            })
    return pd.DataFrame(rows)


def experiment_5(args, data, device):
    """
    Hessian'ın en büyük özdeğeri λ_max (kuvvet yöntemi) eğitim boyunca ölçülür.
    Eğitim Momentum (lr=0.01) ile yapılır: her iki modeli de eğitebilen ortak yöntem.
    λ_max sabit bir eğitim alt kümesinde (ilk N örnek) hesaplanır.
    """
    print("\n" + "=" * 70)
    print("DENEY 5: Hessian spektrumu (kuvvet yöntemi) ve kararlılık sınırı")
    print("=" * 70)
    Xtr, ytr, _, _ = data
    Xs, ys = Xtr[:args.hess_samples], ytr[:args.hess_samples]
    seeds = args.seeds[:args.hess_seeds]
    rows, hist_rows = [], []
    for model_name in MODELS:
        for seed in seeds:
            set_seed(seed)
            model = build_model(model_name).to(device)
            layers = weight_layers(model)
            opt = make_optimizer("Momentum", model.parameters(), DEFAULT_LR["Momentum"],
                                 impl=args.optim_impl)
            gen = torch.Generator().manual_seed(seed)
            for epoch in range(args.hess_epochs + 1):
                train_loss = np.nan
                if epoch > 0:
                    r = train_one_epoch(model, opt, layers, Xtr, ytr, args.batch_size, gen)
                    train_loss = r["train_loss"]
                sync(device)
                t0 = time.perf_counter()
                lam, hist = model_sharpness(model, Xs, ys, n_iter=args.hess_iters,
                                            tol=args.hess_tol, seed=seed)
                sync(device)
                dt = time.perf_counter() - t0
                rows.append(dict(model=model_name, seed=seed, epoch=epoch, train_loss=train_loss,
                                 lambda_max=lam, power_iters=len(hist),
                                 eta_crit_gd=critical_lr(lam),
                                 eta_crit_momentum=critical_lr(lam, momentum=0.9),
                                 time_s=dt))
                if seed == seeds[0]:
                    hist_rows += [dict(model=model_name, epoch=epoch, k=k + 1, lambda_k=v)
                                  for k, v in enumerate(hist)]
                print(f"  {model_name:<10} seed={seed:<4} epoch {epoch}: λ_max={lam:10.4g}  "
                      f"η_kritik(GD)={critical_lr(lam):9.4g}  ({len(hist)} iter, {dt:.1f}s)")
    save_csv(rows, os.path.join(args.results_dir, "exp5_sharpness.csv"))
    save_csv(hist_rows, os.path.join(args.results_dir, "exp5_power_history.csv"))
    print(f"-> {os.path.join(args.results_dir, 'exp5_sharpness.csv')}")


# (etiket, model, optimizer, lr, Deney 1'deki karşılaştırma tabanı (model, optimizer))
ABLATIONS = [
    ("DeepCNN+BN / SGD", "DeepCNN_BN", "SGD", 0.01, ("DeepCNN", "SGD")),
    ("DeepCNN+BN / Momentum", "DeepCNN_BN", "Momentum", 0.01, ("DeepCNN", "Momentum")),
    ("DeepCNN / Adam (düzeltmesiz)", "DeepCNN", "Adam-noBC", 0.001, ("DeepCNN", "Adam")),
    ("ShallowCNN / Adam (düzeltmesiz)", "ShallowCNN", "Adam-noBC", 0.001, ("ShallowCNN", "Adam")),
]


def experiment_6(args, data, device):
    print("\n" + "=" * 70)
    print("DENEY 6: Ablasyon (BatchNorm, Adam yanlılık düzeltmesi)")
    print("=" * 70)
    epoch_rows, run_rows = [], []
    total = len(ABLATIONS) * len(args.seeds)
    k = 0
    for label, model_name, opt, lr, _ in ABLATIONS:
        # Yanlılık düzeltmesiz Adam'ın PyTorch karşılığı yok: daima sıfırdan uygulama
        impl = "manual" if opt == "Adam-noBC" else args.optim_impl
        for seed in args.seeds:
            k += 1
            print(f"[{k}/{total}] {label:<32} seed={seed:<4}")
            e, _, s = run_training(model_name, opt, lr, seed, args.epochs, data,
                                   args.batch_size, device, tag="   ", impl=impl)
            epoch_rows += [dict(r, variant=label) for r in e]
            run_rows.append(dict(s, variant=label))
            save_csv(epoch_rows, os.path.join(args.results_dir, "exp6_epochs.csv"))
            save_csv(run_rows, os.path.join(args.results_dir, "exp6_runs.csv"))

    base_path = os.path.join(args.results_dir, "exp1_runs.csv")
    base = pd.read_csv(base_path) if os.path.exists(base_path) else None
    summary = summarize_exp6(pd.DataFrame(run_rows), pd.DataFrame(epoch_rows), base)
    path = os.path.join(args.results_dir, "exp6_summary.csv")
    summary.to_csv(path, index=False, encoding="utf-8-sig")
    print("\nDeney 6 özeti:")
    print(summary.to_string(index=False))
    print(f"-> {path}")


def summarize_exp6(runs_df, epochs_df, base_df=None):
    """Ablasyon varyantı vs Deney 1 tabanı; aynı seed'lerle eşleştirilmiş t-testi."""
    ett = epochs_to_target(epochs_df.assign(model=epochs_df.variant))
    rows = []
    for label, _, _, _, (bmodel, bopt) in ABLATIONS:
        g = runs_df[runs_df.variant == label].set_index("seed")
        if g.empty:
            continue
        e = ett[ett.model == label].epochs_to_target
        row = {
            "varyant": label,
            "n_seeds": len(g),
            "n_diverged": int(g.status.isin(["nan_inf", "exploded"]).sum()),
            "final_test_acc(%)": fmt_pm(g.final_test_acc * 100, "{:.2f}"),
            "final_train_loss": fmt_pm(g.final_train_loss),
            f"epochs_to_{int(TARGET_ACC * 100)}%": fmt_pm(e, "{:.1f}")
            if np.isfinite(e).any() else "ulaşılamadı",
            "taban": f"{bmodel} / {bopt}",
        }
        if base_df is not None:
            b = base_df[(base_df.model == bmodel) & (base_df.optimizer == bopt)].set_index("seed")
            common = sorted(set(g.index) & set(b.index))
            va = g.loc[common, "final_test_acc"].values * 100
            vb = b.loc[common, "final_test_acc"].values * 100
            row["taban_test_acc(%)"] = fmt_pm(vb, "{:.2f}")
            row["fark(puan)"] = f"{np.nanmean(va - vb):+.2f}"
            ok = len(common) > 1 and np.all(np.isfinite(va - vb)) and np.std(va - vb) > 0
            row["paired_t_p"] = f"{stats.ttest_rel(va, vb).pvalue:.2e}" if ok else "NaN"
        rows.append(row)
    return pd.DataFrame(rows)


def experiment_3(args, data):
    """Otomatik türev vs merkezi sonlu fark: h'ye göre göreli hata (CPU, float32/64)."""
    print("\n" + "=" * 70)
    print("DENEY 3: Gradyan doğrulaması ve sonlu fark hata analizi")
    print("=" * 70)
    _, _, Xte, yte = data
    rows = []
    for model_name in MODELS:
        r = directional_fd_errors(model_name, Xte, yte, default_hs(), seed=args.seeds[0])
        rows += r
        for dtype in ("float64", "float32"):
            sub = [x for x in r if x["dtype"] == dtype]
            best = min(sub, key=lambda x: x["rel_error"])
            print(f"  {model_name:<10} {dtype}: en küçük göreli hata {best['rel_error']:.2e} "
                  f"(h = {best['h']:.1e})")
    path = os.path.join(args.results_dir, "exp3_fd_error.csv")
    save_csv(rows, path)
    print(f"-> {path}")


def experiment_4(args):
    """İterasyon başına süre: CPU ve (varsa) GPU. Rastgele veriyle, ManualAdam ile."""
    print("\n" + "=" * 70)
    print("DENEY 4: CPU / GPU performans karşılaştırması")
    print("=" * 70)
    devices = [torch.device("cpu")]
    if torch.cuda.is_available():
        devices.append(torch.device("cuda"))
    else:
        print("  CUDA bulunamadı: yalnızca CPU ölçülecek.")

    rows = []
    n_warmup, n_iters = 5, args.bench_iters
    for device in devices:
        name = torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"
        for model_name in MODELS:
            set_seed(args.seeds[0])
            model = build_model(model_name).to(device)
            opt = make_optimizer("Adam", model.parameters(), 1e-3, impl=args.optim_impl)
            xb = torch.randn(args.batch_size, 1, 28, 28, device=device)
            yb = torch.randint(0, 10, (args.batch_size,), device=device)

            times = []
            for i in range(n_warmup + n_iters):
                sync(device)
                t0 = time.perf_counter()
                opt.zero_grad(set_to_none=True)
                F.cross_entropy(model(xb), yb).backward()
                opt.step()
                sync(device)
                if i >= n_warmup:  # ısınma iterasyonları (bellek ayırma, çekirdek seçimi) sayılmaz
                    times.append((time.perf_counter() - t0) * 1000)
            t = np.array(times)
            rows.append(dict(device=device.type, device_name=name, model=model_name,
                             batch_size=args.batch_size, n_iters=n_iters,
                             ms_per_iter_mean=t.mean(), ms_per_iter_std=t.std(ddof=1),
                             iters_per_sec=1000 / t.mean()))
            print(f"  {name:<20} {model_name:<10} {t.mean():8.2f} ± {t.std(ddof=1):.2f} ms/iter")

    df = pd.DataFrame(rows)
    if "cuda" in set(df.device):
        cpu = df[df.device == "cpu"].set_index("model").ms_per_iter_mean
        df["speedup_vs_cpu"] = [cpu[m] / v for m, v in zip(df.model, df.ms_per_iter_mean)]
    path = os.path.join(args.results_dir, "exp4_cpu_gpu.csv")
    df.to_csv(path, index=False)
    print(f"-> {path}")


# ---------------------------------------------------------------------------
# Ana program
# ---------------------------------------------------------------------------
def parse_args():
    p = argparse.ArgumentParser(description="CNN optimizer yakınsama/kararlılık deneyleri")
    p.add_argument("--exp", nargs="+", choices=["1", "2", "3", "4", "5", "6", "all"],
                   default=["all"], help="Çalıştırılacak deney(ler), örn. --exp 3 4")
    p.add_argument("--epochs", type=int, default=10, help="Deney 1 ve 6 epoch sayısı")
    p.add_argument("--exp2-epochs", type=int, default=3, help="Deney 2 epoch sayısı")
    p.add_argument("--exp2-seeds", type=int, default=3, help="Deney 2 seed sayısı")
    p.add_argument("--hess-epochs", type=int, default=5, help="Deney 5: kaç epoch boyunca ölçülür")
    p.add_argument("--hess-seeds", type=int, default=3, help="Deney 5 seed sayısı")
    p.add_argument("--hess-samples", type=int, default=1024,
                   help="Deney 5: λ_max'ın hesaplandığı sabit eğitim alt kümesi boyutu")
    p.add_argument("--hess-iters", type=int, default=100, help="Kuvvet yöntemi en fazla iterasyon")
    p.add_argument("--hess-tol", type=float, default=1e-3, help="Kuvvet yöntemi göreli tolerans")
    p.add_argument("--bench-iters", type=int, default=50, help="Deney 4 ölçülen iterasyon sayısı")
    p.add_argument("--optim-impl", choices=["manual", "torch"], default="manual",
                   help="manual: optimizers.py (sıfırdan), torch: PyTorch referansı")
    p.add_argument("--resummarize", action="store_true",
                   help="Eğitim yapmadan mevcut exp1 CSV'lerinden exp1_summary.csv'yi yeniden üret")
    p.add_argument("--batch-size", type=int, default=128)
    p.add_argument("--seeds", type=int, nargs="+", default=SEEDS)
    p.add_argument("--models", nargs="+", default=MODELS, choices=MODELS)
    p.add_argument("--device", default="auto", help="'auto', 'cpu' veya 'cuda'")
    p.add_argument("--data-dir", default="data")
    p.add_argument("--results-dir", default="results")
    p.add_argument("--plots-dir", default="plots")
    p.add_argument("--quick", action="store_true",
                   help="Duman testi: 6000 eğitim / 2000 test örneği, 2 epoch, 2 seed")
    p.add_argument("--no-plot", action="store_true", help="Sonda grafikleri üretme")
    return p.parse_args()


def main():
    args = parse_args()
    if args.resummarize:
        runs = pd.read_csv(os.path.join(args.results_dir, "exp1_runs.csv"))
        epochs = pd.read_csv(os.path.join(args.results_dir, "exp1_epochs.csv"))
        ipe = (int(runs.iters_per_epoch.iloc[0]) if "iters_per_epoch" in runs
               else math.ceil(60000 / args.batch_size))
        write_exp1_summary(runs, epochs, ipe, args.results_dir)
        return

    if args.device == "auto":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(args.device)

    train_subset = test_subset = None
    if args.quick:
        train_subset, test_subset = 6000, 2000
        args.epochs, args.exp2_epochs, args.hess_epochs = 2, 1, 1
        args.seeds = args.seeds[:2]
        print("HIZLI MOD: sonuçlar yalnızca kodun çalıştığını doğrulamak içindir.")

    os.makedirs(args.results_dir, exist_ok=True)
    data = load_fashion_mnist(device, args.data_dir, train_subset, test_subset)

    exps = {"1", "2", "3", "4", "5", "6"} if "all" in args.exp else set(args.exp)
    print(f"Optimizer uygulaması: {args.optim_impl}")
    if exps & {"1", "2", "6"}:
        gpu_warmup(data, device, args.batch_size)
    t0 = time.perf_counter()
    if "1" in exps:
        experiment_1(args, data, device)
    if "2" in exps:
        experiment_2(args, data, device)
    if "3" in exps:
        experiment_3(args, data)
    if "4" in exps:
        experiment_4(args)
    if "5" in exps:
        experiment_5(args, data, device)
    if "6" in exps:
        experiment_6(args, data, device)
    print(f"\nToplam süre: {(time.perf_counter() - t0) / 60:.1f} dk")

    if not args.no_plot:
        import plot_results
        plot_results.main(results_dir=args.results_dir, plots_dir=args.plots_dir)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile plot_results.py
"""
plot_results.py
===============
results/ altındaki CSV dosyalarından 300 DPI akademik grafikler üretir ve
plots/ klasörüne kaydeder:

    loss_convergence.png : Eğitim kaybı vs epoch (log y), ortalama ± std bantları
    gradient_norm.png    : ShallowCNN vs DeepCNN katman bazlı gradyan normları
                           ve ilk/son katman gradyan oranının epoch'larla değişimi
    lr_heatmap.png       : Optimizer x öğrenme oranı kararlılık/doğruluk ısı haritası
    time_vs_acc.png      : Toplam eğitim süresi vs son test doğruluğu
    fd_error.png         : Sonlu fark göreli hatası vs h (kesme/yuvarlama dengesi)
    cpu_gpu.png          : CPU ve GPU'da iterasyon başına süre
    sharpness.png        : Kuvvet yöntemi yakınsaması, λ_max(∇²L) ve η_kritik = 2/λ_max
    ablation.png         : BatchNorm ve Adam yanlılık düzeltmesi ablasyonu

Kullanım:
    python plot_results.py
"""

import os

import matplotlib

matplotlib.use("Agg")  # ekran gerektirmeyen arka uç
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

DPI = 300
OPT_ORDER = ["SGD", "Momentum", "RMSprop", "Adam"]
MODEL_ORDER = ["ShallowCNN", "DeepCNN"]

# Renk körlüğü dostu, sabit sıralı kategorik palet (her optimizer hep aynı renk)
OPT_COLORS = {"SGD": "#2a78d6", "Momentum": "#eb6834", "RMSprop": "#1baf7a", "Adam": "#eda100"}
# İkincil kodlama: renk tek başına kimlik taşımasın diye her optimizer'a ayrı işaretçi
OPT_MARKERS = {"SGD": "o", "Momentum": "s", "RMSprop": "^", "Adam": "D"}
MODEL_STYLE = {
    "ShallowCNN": dict(color="#2a78d6", linestyle="-", marker="o"),
    "DeepCNN": dict(color="#eb6834", linestyle="--", marker="s"),
}

# Nötr mürekkep / ızgara tonları
INK = "#2b2b28"
MUTED = "#6b6a63"
GRID = "#e1e0d9"
AXIS = "#c3c2b7"
SURFACE = "#ffffff"

# Tek tonlu (mavi) sıralı renk skalası: açık = düşük doğruluk, koyu = yüksek
SEQ_BLUE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]


def setup_style():
    plt.rcParams.update({
        "figure.dpi": 110,
        "savefig.dpi": DPI,
        "savefig.bbox": "tight",
        "savefig.facecolor": SURFACE,
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "axes.labelsize": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": AXIS,
        "axes.linewidth": 0.8,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": GRID,
        "grid.linewidth": 0.6,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "xtick.labelcolor": INK,
        "ytick.labelcolor": INK,
        "axes.labelcolor": INK,
        "axes.titlecolor": INK,
        "text.color": INK,
        "legend.frameon": False,
        "lines.linewidth": 2,
    })


def _band(mean, std, floor=1e-12):
    """
    Log ölçekte çizilebilen ± std bandı: [m²/(m+s), m+s].
    Alt sınır, üst sınırın log ölçekte aynası (m/(1+s/m)); s ≪ m iken ≈ m − s olur,
    s ≥ m olduğunda ise m − s ≤ 0'ın yarattığı yapay "dibe inen" bandı önler.
    """
    mean = np.asarray(mean, dtype=float)
    std = np.nan_to_num(np.asarray(std, dtype=float))
    hi = mean + std
    lo = np.maximum(mean * mean / np.maximum(hi, floor), floor)
    return lo, hi


def _save(fig, plots_dir, name):
    path = os.path.join(plots_dir, name)
    fig.savefig(path)
    plt.close(fig)
    print(f"  kaydedildi: {path}")


# ---------------------------------------------------------------------------
# 1) Kayıp yakınsaması
# ---------------------------------------------------------------------------
def plot_loss_convergence(epochs_df, plots_dir):
    df = epochs_df[np.isfinite(epochs_df.train_loss)]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
    for ax, model in zip(axes, MODEL_ORDER):
        sub = df[df.model == model]
        for opt in OPT_ORDER:
            s = sub[sub.optimizer == opt]
            if s.empty:
                continue
            g = s.groupby("epoch")["train_loss"].agg(["mean", "std"]).reset_index()
            lo, hi = _band(g["mean"].values, g["std"].values)
            ax.fill_between(g.epoch, lo, hi, color=OPT_COLORS[opt], alpha=0.18, linewidth=0)
            ax.plot(g.epoch, g["mean"], color=OPT_COLORS[opt], marker=OPT_MARKERS[opt],
                    markersize=5, markeredgecolor=SURFACE, markeredgewidth=0.8, label=opt)
        ax.set_yscale("log")
        ax.set_title(model)
        ax.set_xlabel("Epoch")
        ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
        if sub.empty:
            ax.text(0.5, 0.5, "veri yok", transform=ax.transAxes, ha="center", color=MUTED)
    axes[0].set_ylabel("Eğitim kaybı (log ölçek)")
    handles, labels = axes[0].get_legend_handles_labels()
    if not handles:
        handles, labels = axes[1].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.06))
    fig.text(0.5, -0.04, "Çizgi: seed'ler üzerinden ortalama; gölge: ± 1 standart sapma",
             ha="center", color=MUTED, fontsize=8.5)
    fig.tight_layout()
    _save(fig, plots_dir, "loss_convergence.png")


# ---------------------------------------------------------------------------
# 2) Gradyan normları
# ---------------------------------------------------------------------------
def plot_gradient_norm(layer_df, epochs_df, plots_dir, grad_epoch=1):
    """
    Üst satır : epoch `grad_epoch` boyunca ortalama ||grad||_2, katman sırasına göre
                (girdi -> çıktı). DeepCNN'de ilk katmanlara doğru düşüş = gradyan sönmesi.
    Alt satır : ||g_ilk|| / ||g_son|| oranının epoch'larla değişimi (1'den çok küçük
                ise ilk katmanlar neredeyse öğrenmiyor demektir).
    """
    fig, axes = plt.subplots(2, 4, figsize=(15, 6.8))
    for j, opt in enumerate(OPT_ORDER):
        ax_top, ax_bot = axes[0, j], axes[1, j]
        for model in MODEL_ORDER:
            st = MODEL_STYLE[model]

            # --- katman bazlı norm ---
            s = layer_df[(layer_df.model == model) & (layer_df.optimizer == opt)
                         & (layer_df.epoch == grad_epoch) & np.isfinite(layer_df.grad_norm)]
            if not s.empty:
                g = (s.groupby(["layer_idx", "layer_name"])["grad_norm"]
                      .agg(["mean", "std"]).reset_index().sort_values("layer_idx"))
                x = g.layer_idx.values + 1
                lo, hi = _band(g["mean"].values, g["std"].values)
                ax_top.fill_between(x, lo, hi, color=st["color"], alpha=0.15, linewidth=0)
                ax_top.plot(x, g["mean"], color=st["color"], linestyle=st["linestyle"],
                            marker=st["marker"], markersize=4.5, label=model)

            # --- ilk/son katman oranı ---
            e = epochs_df[(epochs_df.model == model) & (epochs_df.optimizer == opt)].copy()
            e["ratio"] = e.grad_first / e.grad_last
            e = e[np.isfinite(e.ratio) & (e.ratio > 0)]
            if not e.empty:
                g = e.groupby("epoch")["ratio"].agg(["mean", "std"]).reset_index()
                lo, hi = _band(g["mean"].values, g["std"].values)
                ax_bot.fill_between(g.epoch, lo, hi, color=st["color"], alpha=0.15, linewidth=0)
                ax_bot.plot(g.epoch, g["mean"], color=st["color"], linestyle=st["linestyle"],
                            marker=st["marker"], markersize=4.5, label=model)

        ax_top.set_yscale("log")
        ax_top.set_title(opt)
        ax_top.set_xlabel("Katman indeksi (girdi → çıktı)")
        ax_top.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))

        ax_bot.set_yscale("log")
        ax_bot.axhline(1.0, color=AXIS, linestyle=":", linewidth=1.2)
        ax_bot.set_xlabel("Epoch")
        ax_bot.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))

    axes[0, 0].set_ylabel(f"‖∇W‖₂  (epoch {grad_epoch} ortalaması)")
    axes[1, 0].set_ylabel("‖∇W_ilk‖₂ / ‖∇W_son‖₂")
    handles = [Line2D([0], [0], color=MODEL_STYLE[m]["color"], linestyle=MODEL_STYLE[m]["linestyle"],
                      marker=MODEL_STYLE[m]["marker"], label=m) for m in MODEL_ORDER]
    fig.legend(handles=handles, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.04))
    fig.text(0.5, -0.02,
             "Üst: katman bazlı L2 gradyan normu (seed ortalaması ± std). "
             "Alt: ilk/son katman gradyan oranı; 1'in çok altı → gradyan sönmesi.",
             ha="center", color=MUTED, fontsize=8.5)
    fig.tight_layout()
    _save(fig, plots_dir, "gradient_norm.png")


# ---------------------------------------------------------------------------
# 3) Öğrenme oranı ısı haritası
# ---------------------------------------------------------------------------
def plot_lr_heatmap(exp2_df, plots_dir):
    lrs = sorted(exp2_df.lr.unique(), reverse=True)
    opts = [o for o in OPT_ORDER if o in set(exp2_df.optimizer)]
    # Birden çok seed varsa hücre = seed'ler üzerinden özet:
    #   renk/metin: sonlu koşuların ortalama doğruluğu; alt satır: başarısız seed sayıları
    acc = np.full((len(opts), len(lrs)), np.nan)
    status = np.full((len(opts), len(lrs)), "", dtype=object)
    div_ep = np.full((len(opts), len(lrs)), np.nan)
    all_nan = np.zeros((len(opts), len(lrs)), dtype=bool)
    n_seeds = exp2_df.groupby(["optimizer", "lr"]).size().max()

    for (opt, lr), g in exp2_df.groupby(["optimizer", "lr"]):
        if opt not in opts:
            continue
        i, j = opts.index(opt), lrs.index(lr)
        a = g.final_test_acc.values * 100
        acc[i, j] = np.nanmean(a) if np.isfinite(a).any() else np.nan
        n = len(g)
        counts = g.status.value_counts()
        all_nan[i, j] = counts.get("nan_inf", 0) == n
        div_ep[i, j] = g.diverged_epoch.min() if g.diverged_epoch.notna().any() else np.nan
        if n == 1:
            status[i, j] = g.status.iloc[0]
        else:
            parts = [f"{lbl} {counts[s]}/{n}" for s, lbl in
                     (("nan_inf", "NaN"), ("exploded", "patladı"), ("stuck", "takıldı"))
                     if counts.get(s, 0) > 0]
            status[i, j] = "\n".join(parts)

    exploded = all_nan | ~np.isfinite(acc)
    cmap = LinearSegmentedColormap.from_list("seq_blue", SEQ_BLUE)
    cmap.set_bad(SURFACE)

    fig, ax = plt.subplots(figsize=(8.5, 0.75 * len(opts) + 1.8))
    im = ax.imshow(np.ma.masked_where(exploded, acc), cmap=cmap, vmin=0, vmax=100, aspect="auto")

    for i in range(len(opts)):
        for j in range(len(lrs)):
            if exploded[i, j]:
                # Patlayan hücre: taralı, renk skalasının dışında
                ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, facecolor=GRID,
                                       edgecolor="#a8a79d", hatch="///", linewidth=0))
                ep = f"\nepoch {int(div_ep[i, j])}" if np.isfinite(div_ep[i, j]) else ""
                tag = f" {n_seeds}/{n_seeds}" if n_seeds > 1 else ""
                ax.text(j, i, f"NaN/Inf{tag}{ep}", ha="center", va="center", fontsize=8.5,
                        fontweight="bold", color=INK)
            else:
                txt = f"{acc[i, j]:.1f}%"
                if status[i, j] == "stuck":
                    txt += "\n(takıldı)"
                elif status[i, j] == "exploded":
                    txt += "\n(kayıp patladı)"
                elif status[i, j] not in ("", "stable"):
                    txt += "\n" + status[i, j]
                ax.text(j, i, txt, ha="center", va="center", fontsize=8 if n_seeds > 1 else 9,
                        color=SURFACE if acc[i, j] > 55 else INK)

    ax.set_xticks(range(len(lrs)))
    ax.set_xticklabels([f"{lr:g}" for lr in lrs])
    ax.set_yticks(range(len(opts)))
    ax.set_yticklabels(opts)
    ax.set_xlabel("Öğrenme oranı (lr)")
    ax.set_title("DeepCNN: öğrenme oranı – optimizer kararlılık haritası")

    # Hücreler arasında beyaz boşluk
    ax.set_xticks(np.arange(-0.5, len(lrs), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(opts), 1), minor=True)
    ax.grid(which="minor", color=SURFACE, linewidth=2.5)
    ax.grid(which="major", visible=False)
    ax.tick_params(which="both", length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)

    cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    cb.set_label("Son test doğruluğu (%)")
    cb.outline.set_visible(False)
    fig.text(0.5, -0.03,
             f"{n_seeds} seed; hücre değeri: sonlu koşuların ortalama son test doğruluğu.  "
             "Taralı: tüm seed'lerde NaN/Inf (kayan nokta taşması).\n'patladı': sonlu ama "
             "eğitim kaybı > 10·ln10.  'takıldı': kayıp sonlu ve küçük, doğruluk şans "
             "düzeyinde (< %15).",
             ha="center", color=MUTED, fontsize=8.5)
    fig.tight_layout()
    _save(fig, plots_dir, "lr_heatmap.png")


# ---------------------------------------------------------------------------
# 4) Süre vs doğruluk
# ---------------------------------------------------------------------------
def plot_time_vs_acc(runs_df, plots_dir):
    fig, ax = plt.subplots(figsize=(7.5, 5))
    for model in MODEL_ORDER:
        for opt in OPT_ORDER:
            s = runs_df[(runs_df.model == model) & (runs_df.optimizer == opt)]
            s = s[np.isfinite(s.final_test_acc)]
            if s.empty:
                continue
            c = OPT_COLORS[opt]
            face = c if model == "ShallowCNN" else SURFACE  # dolu = sığ, boş = derin
            # Tek tek seed'ler (soluk)
            ax.scatter(s.total_time, s.final_test_acc * 100, s=14, color=c, alpha=0.3,
                       marker=OPT_MARKERS[opt], linewidths=0)
            # Ortalama ± std
            ax.errorbar(s.total_time.mean(), s.final_test_acc.mean() * 100,
                        xerr=np.nan_to_num(s.total_time.std()),
                        yerr=np.nan_to_num(s.final_test_acc.std() * 100),
                        fmt=OPT_MARKERS[opt], markersize=9, markerfacecolor=face,
                        markeredgecolor=c, markeredgewidth=2, ecolor=c, elinewidth=1.2,
                        capsize=3, zorder=3)

    ax.set_xlabel("Toplam eğitim süresi (s)")
    ax.set_ylabel("Son test doğruluğu (%)")
    ax.set_title("Eğitim süresi – test doğruluğu")

    opt_handles = [Line2D([0], [0], marker=OPT_MARKERS[o], color="none", markerfacecolor=OPT_COLORS[o],
                          markeredgecolor=OPT_COLORS[o], markersize=8, label=o) for o in OPT_ORDER]
    model_handles = [
        Line2D([0], [0], marker="o", color="none", markerfacecolor=MUTED, markeredgecolor=MUTED,
               markersize=8, label="ShallowCNN (dolu)"),
        Line2D([0], [0], marker="o", color="none", markerfacecolor=SURFACE, markeredgecolor=MUTED,
               markeredgewidth=2, markersize=8, label="DeepCNN (boş)"),
    ]
    leg1 = ax.legend(handles=opt_handles, title="Optimizer", loc="lower right")
    ax.add_artist(leg1)
    ax.legend(handles=model_handles, title="Model", loc="center left")
    fig.text(0.5, -0.03, "Büyük işaretçi: seed ortalaması ± std;  küçük soluk noktalar: tek tek seed'ler",
             ha="center", color=MUTED, fontsize=8.5)
    fig.tight_layout()
    _save(fig, plots_dir, "time_vs_acc.png")


# ---------------------------------------------------------------------------
# 5) Sonlu fark hata analizi
# ---------------------------------------------------------------------------
DTYPE_STYLE = {
    "float64": dict(color="#2a78d6", marker="o", eps=np.finfo(np.float64).eps),
    "float32": dict(color="#eb6834", marker="s", eps=np.finfo(np.float32).eps),
}


def plot_fd_error(fd_df, plots_dir):
    models = [m for m in MODEL_ORDER if m in set(fd_df.model)]
    fig, axes = plt.subplots(1, len(models), figsize=(5.2 * len(models), 4.3), sharey=True,
                             squeeze=False)
    for ax, model in zip(axes[0], models):
        sub = fd_df[fd_df.model == model]
        for dtype, st in DTYPE_STYLE.items():
            s = sub[sub.dtype == dtype].sort_values("h")
            if s.empty:
                continue
            err = np.maximum(s.rel_error.values, 1e-17)  # log eksende 0 çizilemez
            ax.plot(s.h, err, color=st["color"], marker=st["marker"], markersize=4.5, label=dtype)
            # Kuramsal en iyi adım h* ~ ε^(1/3)
            ax.axvline(st["eps"] ** (1 / 3), color=st["color"], linestyle=":", linewidth=1.2)

        # Eğim kılavuzları (float64 eğrisine sabitlenmiş): sağda ∝ h², solda ∝ 1/h
        s64 = sub[sub.dtype == "float64"].sort_values("h")
        if len(s64) > 3:
            h = s64.h.values
            e = np.maximum(s64.rel_error.values, 1e-17)
            i_r = np.argmin(np.abs(np.log10(h) + 2))      # h ≈ 1e-2
            hr = np.logspace(-4, 0, 20)
            ax.plot(hr, e[i_r] * (hr / h[i_r]) ** 2, color=MUTED, linestyle="--", linewidth=1)
            ax.text(hr[-6], e[i_r] * (hr[-6] / h[i_r]) ** 2 * 3, "∝ h²  (kesme)",
                    color=MUTED, fontsize=8.5, ha="right")
            i_l = np.argmin(np.abs(np.log10(h) + 10))     # h ≈ 1e-10
            hl = np.logspace(-12, -7, 20)
            ax.plot(hl, e[i_l] * (h[i_l] / hl), color=MUTED, linestyle="--", linewidth=1)
            ax.text(hl[8], e[i_l] * (h[i_l] / hl[8]) / 6, "∝ ε/h  (yuvarlama)",
                    color=MUTED, fontsize=8.5, ha="left")

        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_title(model)
        ax.set_xlabel("Adım boyu h")
    axes[0, 0].set_ylabel("Göreli hata  |D_h − ∇L·v| / |∇L·v|")
    axes[0, 0].legend(title="Aritmetik", loc="lower left")
    fig.text(0.5, -0.03,
             "Merkezi fark D_h = [L(θ+hv) − L(θ−hv)] / 2h, referans: float64 otomatik türev.  "
             "Noktalı dikey çizgiler: kuramsal en iyi adım h* ≈ ε^(1/3).",
             ha="center", color=MUTED, fontsize=8.5)
    fig.tight_layout()
    _save(fig, plots_dir, "fd_error.png")


# ---------------------------------------------------------------------------
# 6) CPU / GPU
# ---------------------------------------------------------------------------
def plot_cpu_gpu(bench_df, plots_dir):
    devices = list(dict.fromkeys(bench_df.device))           # cpu, (cuda)
    names = {d: bench_df[bench_df.device == d].device_name.iloc[0] for d in devices}
    colors = {"cpu": "#2a78d6", "cuda": "#eb6834"}
    models = [m for m in MODEL_ORDER if m in set(bench_df.model)]
    x = np.arange(len(models))
    width = 0.8 / len(devices)

    fig, ax = plt.subplots(figsize=(6.5, 4.3))
    for k, d in enumerate(devices):
        s = bench_df[bench_df.device == d].set_index("model").loc[models]
        pos = x + (k - (len(devices) - 1) / 2) * width
        ax.bar(pos, s.ms_per_iter_mean, width=width - 0.03, yerr=s.ms_per_iter_std,
               color=colors.get(d, MUTED), label=names[d], capsize=3,
               error_kw=dict(ecolor=INK, elinewidth=1))
        for p, v, sd, row in zip(pos, s.ms_per_iter_mean, s.ms_per_iter_std, s.itertuples()):
            label = f"{v:.1f} ms"
            if d == "cuda" and "speedup_vs_cpu" in s.columns:
                label += f"\n×{row.speedup_vs_cpu:.1f}"
            # Etiket hata çubuğunun üstüne yazılır (çubukla çakışmasın)
            ax.text(p, v + np.nan_to_num(sd) + 1.5, label, ha="center", va="bottom",
                    fontsize=8.5, color=INK)
    ax.set_xticks(x)
    ax.set_xticklabels(models)
    ax.set_ylabel("İterasyon başına süre (ms)")
    ax.set_title("CPU – GPU karşılaştırması (ileri + geri + Adam adımı)")
    ax.grid(axis="x", visible=False)
    ax.legend(loc="upper left")
    ax.margins(y=0.2)
    fig.tight_layout()
    _save(fig, plots_dir, "cpu_gpu.png")


# ---------------------------------------------------------------------------
# 7) Hessian spektrumu: kuvvet yöntemi yakınsaması, λ_max ve η_kritik
# ---------------------------------------------------------------------------
def _exp2_boundary(exp2_df, opt="SGD"):
    """Deney 2'de tüm seed'lerde kararlı en büyük lr ve en az bir seed'de ıraksayan en küçük lr."""
    if exp2_df is None:
        return None, None
    g = exp2_df[exp2_df.optimizer == opt]
    ok = [lr for lr, s in g.groupby("lr") if (s.status == "stable").all()]
    bad = [lr for lr, s in g.groupby("lr") if s.status.isin(["nan_inf", "exploded"]).any()]
    return (max(ok) if ok else None), (min(bad) if bad else None)


def plot_sharpness(sharp_df, hist_df, plots_dir, exp2_df=None):
    models = [m for m in MODEL_ORDER if m in set(sharp_df.model)]
    # Kuvvet yöntemi mutlak değerce en büyük özdeğere yakınsar. Düzlük bölgesinde (ör.
    # DeepCNN'in başlangıç platosu) bu negatif bir özdeğer olabilir; o zaman λ_max
    # bilinmez ve 2/λ_max tanımsızdır. Bu ölçümler (b) ve (c)'den çıkarılır, dipnotta sayılır.
    neg = sharp_df[sharp_df.lambda_max <= 0]
    sharp_pos = sharp_df[sharp_df.lambda_max > 0]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))

    # (a) Kuvvet yönteminin yakınsaması: göreli hata |λ_k − λ_son| / |λ_son|
    ax = axes[0]
    if hist_df is not None and not hist_df.empty:
        last_ep = hist_df.epoch.max()
        for model in models:
            st = MODEL_STYLE[model]
            for ep, ls in ((0, ":"), (last_ep, st["linestyle"])):
                h = hist_df[(hist_df.model == model) & (hist_df.epoch == ep)].sort_values("k")
                if len(h) < 2:
                    continue
                lam = h.lambda_k.values
                err = np.abs(lam[:-1] - lam[-1]) / max(abs(lam[-1]), 1e-30)
                ax.plot(h.k.values[:-1], np.maximum(err, 1e-16), color=st["color"],
                        linestyle=ls, linewidth=1.6, label=f"{model}, epoch {ep}")
        ax.set_yscale("log")
        ax.set_xlabel("Kuvvet yöntemi iterasyonu k")
        ax.set_ylabel("|λ_k − λ_son| / |λ_son|")
        ax.set_title("(a) Kuvvet yönteminin yakınsaması")
        ax.legend(fontsize=8)

    # (b) λ_max'ın eğitim boyunca değişimi
    ax = axes[1]
    for model in models:
        g = sharp_pos[sharp_pos.model == model].groupby("epoch").lambda_max
        m, s = g.mean(), g.std().fillna(0)
        st = MODEL_STYLE[model]
        ax.plot(m.index, m.values, color=st["color"], linestyle=st["linestyle"],
                marker=st["marker"], markersize=5, label=model)
        lo, hi = _band(m.values, s.values)
        ax.fill_between(m.index, lo, hi, color=st["color"], alpha=0.15, linewidth=0)
    ax.set_yscale("log")
    ax.set_xlabel("Epoch (Momentum, η = 0.01 ile eğitim)")
    ax.set_ylabel("λ_max(∇²L)")
    ax.set_title("(b) Hessian'ın en büyük özdeğeri")
    ax.legend()

    # (c) Teorik kararlılık sınırı η_kritik = 2/λ_max ve Deney 2'deki deneysel sınır
    ax = axes[2]
    for model in models:
        g = sharp_pos[sharp_pos.model == model].groupby("epoch").eta_crit_gd
        m, s = g.mean(), g.std().fillna(0)
        st = MODEL_STYLE[model]
        ax.plot(m.index, m.values, color=st["color"], linestyle=st["linestyle"],
                marker=st["marker"], markersize=5, label=f"{model}: 2/λ_max")
    ok, bad = _exp2_boundary(exp2_df, "SGD")
    if ok is not None:
        ax.axhline(ok, color=OPT_COLORS["SGD"], linestyle=":", linewidth=1.3,
                   label=f"Deney 2 (DeepCNN, SGD): kararlı η = {ok:g}")
    if bad is not None:
        ax.axhline(bad, color="#c0392b", linestyle=":", linewidth=1.3,
                   label=f"Deney 2 (DeepCNN, SGD): ıraksayan η = {bad:g}")
    ax.set_yscale("log")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("η_kritik (SGD)")
    ax.set_title("(c) Teorik kararlılık sınırı η < 2/λ_max")
    ax.legend(fontsize=8)

    note = ("λ_max: Hessian–vektör çarpımı + kuvvet yöntemi, sabit 1024 örneklik eğitim alt "
            "kümesi, seed ortalaması ± std.  Yerel ikinci derece modelde GD yalnızca "
            "η < 2/λ_max için kararlıdır.")
    if not neg.empty:
        which = ", ".join(f"{r.model} seed {r.seed} epoch {r.epoch}" for r in neg.itertuples())
        note += (f"\n(b)–(c): kuvvet yönteminin negatif baskın özdeğer verdiği {len(neg)} ölçüm "
                 f"hariç tutuldu ({which}).")
    fig.text(0.5, -0.04 if neg.empty else -0.08, note, ha="center", color=MUTED, fontsize=8.5)
    fig.tight_layout()
    _save(fig, plots_dir, "sharpness.png")


# ---------------------------------------------------------------------------
# 8) Ablasyon: BatchNorm ve Adam yanlılık düzeltmesi
# ---------------------------------------------------------------------------
def _mean_curve(df, col="train_loss"):
    g = df.groupby("epoch")[col]
    return g.mean(), g.std().fillna(0)


def plot_ablation(exp1_epochs, exp6_epochs, plots_dir):
    panels = [
        ("(a) SGD: BatchNorm ablasyonu",
         [("DeepCNN (BN yok)", exp1_epochs, dict(model="DeepCNN", optimizer="SGD"), "#eb6834", "--"),
          ("DeepCNN + BN", exp6_epochs, dict(variant="DeepCNN+BN / SGD"), "#2a78d6", "-")]),
        ("(b) Momentum: BatchNorm ablasyonu",
         [("DeepCNN (BN yok)", exp1_epochs, dict(model="DeepCNN", optimizer="Momentum"), "#eb6834", "--"),
          ("DeepCNN + BN", exp6_epochs, dict(variant="DeepCNN+BN / Momentum"), "#2a78d6", "-")]),
        ("(c) Adam: yanlılık düzeltmesi ablasyonu",
         [("ShallowCNN, düzeltmeli", exp1_epochs, dict(model="ShallowCNN", optimizer="Adam"), "#2a78d6", "-"),
          ("ShallowCNN, düzeltmesiz", exp6_epochs, dict(variant="ShallowCNN / Adam (düzeltmesiz)"),
           "#2a78d6", ":"),
          ("DeepCNN, düzeltmeli", exp1_epochs, dict(model="DeepCNN", optimizer="Adam"), "#eb6834", "-"),
          ("DeepCNN, düzeltmesiz", exp6_epochs, dict(variant="DeepCNN / Adam (düzeltmesiz)"),
           "#eb6834", ":")]),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
    for ax, (title, lines) in zip(axes, panels):
        for label, df, filt, color, ls in lines:
            if df is None:
                continue
            sub = df
            for k, v in filt.items():
                if k not in sub.columns:
                    sub = sub.iloc[0:0]
                    break
                sub = sub[sub[k] == v]
            if sub.empty:
                continue
            m, s = _mean_curve(sub)
            ax.plot(m.index, m.values, color=color, linestyle=ls, marker="o", markersize=3.5,
                    label=label)
            lo, hi = _band(m.values, s.values)
            ax.fill_between(m.index, lo, hi, color=color, alpha=0.12, linewidth=0)
        ax.set_yscale("log")
        ax.set_xlabel("Epoch")
        ax.set_title(title)
        ax.legend(fontsize=8)
    axes[0].set_ylabel("Eğitim kaybı (log ölçek)")
    fig.text(0.5, -0.03, "Aynı 5 seed (aynı başlangıç ağırlıkları ve mini-batch sırası); "
             "çizgi: ortalama, gölge: ± 1 std.", ha="center", color=MUTED, fontsize=8.5)
    fig.tight_layout()
    _save(fig, plots_dir, "ablation.png")


# ---------------------------------------------------------------------------
def _read(results_dir, name):
    path = os.path.join(results_dir, name)
    if not os.path.exists(path):
        print(f"  atlandı: {path} bulunamadı")
        return None
    return pd.read_csv(path)


def main(results_dir="results", plots_dir="plots"):
    setup_style()
    os.makedirs(plots_dir, exist_ok=True)
    print("\nGrafikler üretiliyor...")

    epochs = _read(results_dir, "exp1_epochs.csv")
    layers = _read(results_dir, "exp1_layer_grads.csv")
    runs = _read(results_dir, "exp1_runs.csv")
    exp2 = _read(results_dir, "exp2_lr_stability.csv")
    fd = _read(results_dir, "exp3_fd_error.csv")
    bench = _read(results_dir, "exp4_cpu_gpu.csv")

    if fd is not None:
        plot_fd_error(fd, plots_dir)
    if bench is not None:
        plot_cpu_gpu(bench, plots_dir)
    if epochs is not None:
        plot_loss_convergence(epochs, plots_dir)
    if epochs is not None and layers is not None:
        plot_gradient_norm(layers, epochs, plots_dir)
    if exp2 is not None:
        plot_lr_heatmap(exp2, plots_dir)
    if runs is not None:
        plot_time_vs_acc(runs, plots_dir)

    sharp = _read(results_dir, "exp5_sharpness.csv")
    hist = _read(results_dir, "exp5_power_history.csv")
    if sharp is not None:
        plot_sharpness(sharp, hist, plots_dir, exp2)
    exp6 = _read(results_dir, "exp6_epochs.csv")
    if exp6 is not None:
        plot_ablation(epochs, exp6, plots_dir)


if __name__ == "__main__":
    import argparse

    ap = argparse.ArgumentParser(description="Sonuç CSV'lerinden grafik üretir")
    ap.add_argument("--results-dir", default="results")
    ap.add_argument("--plots-dir", default="plots")
    a = ap.parse_args()
    main(results_dir=a.results_dir, plots_dir=a.plots_dir)


In [ ]:
%%writefile tests.py
"""
tests.py
========
Projenin birim testleri (unittest). Veri indirmez; sentetik veriyle CPU'da
yaklaşık 1 dakikada çalışır.

    python -m unittest tests -v

Test grupları ve beklenen çıktılar:
  1. Modeller         : çıktı boyutu (B, 10); ShallowCNN 3, DeepCNN 8 konvolüsyon;
                        DeepCNN'de BatchNorm yok.
  2. Optimizer formülü: elle hesaplanan skaler güncellemelerle birebir eşleşme
                        (float64, 12 ondalık basamak).
  3. Referans eşleşme : sıfırdan yazılan optimizer'lar 20 adım sonra PyTorch
                        referansıyla aynı parametreleri üretir (|fark| < 1e-5).
  4. Determinizm      : aynı seed -> bit düzeyinde aynı kayıp dizisi;
                        farklı seed -> farklı kayıp dizisi.
  5. Patlama tespiti  : NaN içeren girdi ilk iterasyonda 'nan_inf' olarak işaretlenir;
                        sonlu ama devasa kayıp 'exploded', şans düzeyi 'stuck' sayılır.
  6. Gradyan normu    : eğitim döngüsünün ölçtüğü ||∇W||_2 elle hesaplananla eşleşir.
  7. Sonlu fark       : float64'te otomatik türev ile merkezi fark uyumlu (< 1e-5);
                        çok küçük h'de float32 yuvarlama hatası baskın (> 0.1).
  8. Kuvvet yöntemi   : özdeğerleri bilinen ikinci derece fonksiyonda λ_max = 5 (8 basamak),
                        geometrik yakınsama; Hv açık Hessian ile eşleşir; η_kritik = 2(1+μ)/λ.
  9. Ablasyon         : DeepCNN_BN'de 8 BatchNorm; düzeltmesiz Adam'ın ilk adımı
                        η(1−β1)/√(1−β2) ≈ 3.16η.
"""

import copy
import math
import unittest

import torch
import torch.nn as nn
import torch.nn.functional as F

from experiments import (DEFAULT_LR, OPTIMIZERS, classify_status, run_training, set_seed,
                         train_one_epoch)
from gradient_check import directional_fd_errors
from hessian import critical_lr, hvp, model_sharpness, top_eigenvalue
from models import DeepCNN, DeepCNN_BN, ShallowCNN, build_model, weight_layers
from optimizers import ManualAdam, ManualRMSprop, ManualSGD, make_optimizer


def synthetic_data(n=256, seed=0):
    """Rastgele 1x28x28 görüntüler ve 10 sınıflı etiketler (indirme gerektirmez)."""
    g = torch.Generator().manual_seed(seed)
    X = torch.randn(n, 1, 28, 28, generator=g)
    y = torch.randint(0, 10, (n,), generator=g)
    return X, y


def scalar_param(value=1.0):
    return torch.tensor([value], dtype=torch.float64, requires_grad=True)


# ---------------------------------------------------------------------------
class TestModels(unittest.TestCase):
    def test_output_shape(self):
        x = torch.zeros(4, 1, 28, 28)
        for cls in (ShallowCNN, DeepCNN):
            self.assertEqual(tuple(cls()(x).shape), (4, 10))

    def test_layer_counts_and_order(self):
        for name, n_conv in (("ShallowCNN", 3), ("DeepCNN", 8)):
            layers = weight_layers(build_model(name))
            self.assertEqual(sum(isinstance(m, nn.Conv2d) for _, m in layers), n_conv)
            self.assertEqual(layers[0][0], "conv1")   # ilk katman = girdi tarafı
            self.assertEqual(layers[-1][0], "fc2")    # son katman = çıktı tarafı

    def test_deep_has_no_batchnorm(self):
        self.assertFalse(any(isinstance(m, nn.modules.batchnorm._BatchNorm)
                             for m in DeepCNN().modules()))


# ---------------------------------------------------------------------------
class TestOptimizerFormulas(unittest.TestCase):
    """Elle hesaplanan beklenen değerler (θ0 = 1)."""

    def _step(self, opt, p, g):
        p.grad = torch.tensor([g], dtype=torch.float64)
        opt.step()
        return p.item()

    def test_sgd(self):
        p = scalar_param()
        # θ1 = 1 − 0.1·2 = 0.8
        self.assertAlmostEqual(self._step(ManualSGD([p], lr=0.1), p, 2.0), 0.8, places=12)

    def test_momentum(self):
        p = scalar_param()
        opt = ManualSGD([p], lr=0.1, momentum=0.9)
        # v1 = 2           -> θ1 = 1 − 0.1·2 = 0.8
        self.assertAlmostEqual(self._step(opt, p, 2.0), 0.8, places=12)
        # v2 = 0.9·2 + 1 = 2.8 -> θ2 = 0.8 − 0.28 = 0.52
        self.assertAlmostEqual(self._step(opt, p, 1.0), 0.52, places=12)

    def test_rmsprop(self):
        p = scalar_param()
        lr, a, eps, g = 0.01, 0.99, 1e-8, 2.0
        # s1 = (1−α)·g² = 0.04  -> θ1 = 1 − η·g/(√0.04 + ε) ≈ 0.9
        expected = 1 - lr * g / (math.sqrt((1 - a) * g * g) + eps)
        self.assertAlmostEqual(self._step(ManualRMSprop([p], lr=lr, alpha=a, eps=eps), p, g),
                               expected, places=12)

    def test_adam_two_steps(self):
        p = scalar_param()
        lr, b1, b2, eps = 0.001, 0.9, 0.999, 1e-8
        opt = ManualAdam([p], lr=lr, betas=(b1, b2), eps=eps)
        theta, m, v = 1.0, 0.0, 0.0
        for t, g in enumerate((2.0, -1.0), start=1):
            m = b1 * m + (1 - b1) * g
            v = b2 * v + (1 - b2) * g * g
            m_hat, v_hat = m / (1 - b1 ** t), v / (1 - b2 ** t)
            theta -= lr * m_hat / (math.sqrt(v_hat) + eps)
            self.assertAlmostEqual(self._step(opt, p, g), theta, places=12)
        # İlk adımda m̂/√v̂ = sign(g) olduğundan θ1 ≈ 1 − η (Adam'ın ölçek bağımsızlığı)


class TestOptimizersMatchTorch(unittest.TestCase):
    def test_manual_equals_torch_reference(self):
        X, y = synthetic_data(640)
        for name in OPTIMIZERS:
            with self.subTest(optimizer=name):
                set_seed(0)
                model_a = ShallowCNN()
                model_b = copy.deepcopy(model_a)
                opt_a = make_optimizer(name, model_a.parameters(), DEFAULT_LR[name], impl="manual")
                opt_b = make_optimizer(name, model_b.parameters(), DEFAULT_LR[name], impl="torch")
                for k in range(20):
                    xb, yb = X[32 * k:32 * (k + 1)], y[32 * k:32 * (k + 1)]
                    for model, opt in ((model_a, opt_a), (model_b, opt_b)):
                        opt.zero_grad()
                        F.cross_entropy(model(xb), yb).backward()
                        opt.step()
                max_diff = max((pa - pb).abs().max().item()
                               for pa, pb in zip(model_a.parameters(), model_b.parameters()))
                self.assertLess(max_diff, 1e-5, f"{name}: en büyük fark {max_diff:.2e}")


# ---------------------------------------------------------------------------
class TestDeterminism(unittest.TestCase):
    def _losses(self, seed):
        X, y = synthetic_data(256)
        data = (X, y, X[:128], y[:128])
        rows, _, _ = run_training("ShallowCNN", "Adam", 1e-3, seed, epochs=2, data=data,
                                  batch_size=64, device=torch.device("cpu"))
        return [r["train_loss"] for r in rows]

    def test_same_seed_identical(self):
        self.assertEqual(self._losses(42), self._losses(42))

    def test_different_seed_differs(self):
        self.assertNotEqual(self._losses(42), self._losses(7))


class TestDivergenceDetection(unittest.TestCase):
    def test_nan_input_detected_at_first_iteration(self):
        X, y = synthetic_data(128)
        X[0, 0, 0, 0] = float("nan")
        set_seed(0)
        model = ShallowCNN()
        opt = make_optimizer("SGD", model.parameters(), 0.01)
        r = train_one_epoch(model, opt, weight_layers(model), X, y, batch_size=128,
                            generator=torch.Generator().manual_seed(0))
        self.assertTrue(r["diverged"])
        self.assertEqual(r["diverged_iter"], 1)

    def test_run_training_reports_nan_inf(self):
        X, y = synthetic_data(128)
        X[:] = float("inf")
        _, _, summary = run_training("DeepCNN", "Adam", 1e-3, 0, epochs=2, data=(X, y, X, y),
                                     batch_size=64, device=torch.device("cpu"))
        self.assertEqual(summary["status"], "nan_inf")
        self.assertEqual(summary["diverged_epoch"], 1)


class TestAblationComponents(unittest.TestCase):
    def test_deep_bn_model(self):
        m = DeepCNN_BN()
        self.assertEqual(tuple(m(torch.zeros(4, 1, 28, 28)).shape), (4, 10))
        n_bn = sum(isinstance(x, nn.BatchNorm2d) for x in m.modules())
        self.assertEqual(n_bn, 8)                       # her konvolüsyondan sonra bir BN
        # Ağırlık katmanları DeepCNN ile aynı (BN hariç): gradyan ölçümleri karşılaştırılabilir
        self.assertEqual([n for n, _ in weight_layers(m)], [n for n, _ in weight_layers(DeepCNN())])

    def test_adam_without_bias_correction_first_step(self):
        # Düzeltmesiz ilk adım: Δθ = −η (1−β1) g / (√((1−β2) g²) + ε) ≈ −η (1−β1)/√(1−β2)
        p = scalar_param()
        lr, b1, b2, eps, g = 0.001, 0.9, 0.999, 1e-8, 2.0
        opt = ManualAdam([p], lr=lr, betas=(b1, b2), eps=eps, bias_correction=False)
        p.grad = torch.tensor([g], dtype=torch.float64)
        opt.step()
        expected = 1 - lr * (1 - b1) * g / (math.sqrt((1 - b2) * g * g) + eps)
        self.assertAlmostEqual(p.item(), expected, places=12)
        # Adım büyüklüğü düzeltmeli Adam'ınkinin (≈ η) yaklaşık 3.16 katı
        self.assertAlmostEqual((1 - p.item()) / lr, (1 - b1) / math.sqrt(1 - b2), places=5)


class TestPowerIteration(unittest.TestCase):
    """Kuvvet yöntemi, özdeğerleri bilinen bir ikinci derece fonksiyonda sınanır."""

    def test_quadratic_known_spectrum(self):
        # L(x) = ½ xᵀ A x,  A = Q diag(λ) Qᵀ  ⇒  ∇²L = A,  λ_max = 5
        g = torch.Generator().manual_seed(0)
        lams = torch.tensor([5.0, 3.0, 1.0, 0.5, 0.1], dtype=torch.float64)
        Q, _ = torch.linalg.qr(torch.randn(5, 5, generator=g, dtype=torch.float64))
        A = Q @ torch.diag(lams) @ Q.T
        x = torch.randn(5, generator=g, dtype=torch.float64, requires_grad=True)
        lam, hist = top_eigenvalue(lambda: 0.5 * x @ A @ x, [x], n_iter=200, tol=1e-12)
        self.assertAlmostEqual(lam, 5.0, places=8)
        # Rayleigh bölümü hatası ~ (λ2/λ1)^{2k} = 0.36^k: geometrik (doğrusal) yakınsama.
        # Bu oran yalnızca ASİMPTOTİK bölgede geçerlidir: ilk iterasyonlarda v0'ın küçük
        # özdeğer bileşenleri baskındır, sonda ise hata yuvarlama tabanına oturur. Bu yüzden
        # hız, 1e-10 < hata < 1e-3 penceresinde ardışık oranların geometrik ortalamasıyla
        # ölçülür (A pozitif tanımlı olduğundan λ_k monoton artar; pencere kesintisizdir).
        err = [abs(h - 5.0) for h in hist]
        win = [e for e in err if 1e-10 < e < 1e-3]
        self.assertGreaterEqual(len(win), 5)
        rate = (win[-1] / win[0]) ** (1.0 / (len(win) - 1))
        self.assertAlmostEqual(rate, (3.0 / 5.0) ** 2, delta=0.05)

    def test_hvp_matches_explicit_hessian(self):
        g = torch.Generator().manual_seed(1)
        A = torch.randn(4, 4, generator=g, dtype=torch.float64)
        A = A + A.T
        x = torch.randn(4, generator=g, dtype=torch.float64, requires_grad=True)
        v = torch.randn(4, generator=g, dtype=torch.float64)
        (hv,) = hvp(lambda: 0.5 * x @ A @ x + (x ** 3).sum(), [x], [v])
        H = A + torch.diag(6 * x.detach())               # ∇²(Σx³) = diag(6x)
        self.assertTrue(torch.allclose(hv, H @ v, atol=1e-12))

    def test_critical_lr(self):
        self.assertAlmostEqual(critical_lr(4.0), 0.5)
        self.assertAlmostEqual(critical_lr(4.0, momentum=0.9), 0.95)
        self.assertEqual(critical_lr(0.0), float("inf"))

    def test_model_sharpness_positive(self):
        X, y = synthetic_data(64)
        set_seed(0)
        lam, hist = model_sharpness(ShallowCNN().double(), X.double(), y, n_iter=30)
        self.assertTrue(math.isfinite(lam) and lam > 0)
        self.assertGreater(len(hist), 1)


class TestStatusClassification(unittest.TestCase):
    def test_categories(self):
        self.assertEqual(classify_status("nan_inf", [float("nan")], float("nan")), "nan_inf")
        self.assertEqual(classify_status("stable", [3e14], 0.10), "exploded")  # sonlu patlama
        self.assertEqual(classify_status("stable", [2.30, 2.30], 0.10), "stuck")
        self.assertEqual(classify_status("stable", [1e5, 0.6], 0.80), "stable")  # toparlandı


class TestGradientNorm(unittest.TestCase):
    def test_measured_norm_matches_manual(self):
        X, y = synthetic_data(64)
        set_seed(0)
        model = ShallowCNN()
        ref = copy.deepcopy(model)
        # Beklenen: tüm veri tek batch iken ||∇W||_2 doğrudan hesaplanır
        F.cross_entropy(ref(X), y).backward()
        expected = [m.weight.grad.norm(2).item() for _, m in weight_layers(ref)]

        opt = make_optimizer("SGD", model.parameters(), 1e-3)
        r = train_one_epoch(model, opt, weight_layers(model), X, y, batch_size=64,
                            generator=torch.Generator().manual_seed(0))
        for got, exp in zip(r["grad_norms"], expected):
            self.assertAlmostEqual(got / exp, 1.0, places=4)


class TestFiniteDifference(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        X, y = synthetic_data(16)
        rows = directional_fd_errors("ShallowCNN", X, y, hs=[1e-1, 1e-4, 1e-5, 1e-6, 1e-10],
                                     batch=8)
        cls.err = {(r["dtype"], r["h"]): r["rel_error"] for r in rows}

    def test_float64_autograd_matches_fd(self):
        best = min(self.err[("float64", h)] for h in (1e-4, 1e-5, 1e-6))
        self.assertLess(best, 1e-5)

    def test_truncation_error_grows_with_h(self):
        # Büyük h'de kesme hatası O(h²) baskın
        self.assertGreater(self.err[("float64", 1e-1)], self.err[("float64", 1e-5)])

    def test_float32_roundoff_dominates_small_h(self):
        # h = 1e-10'da θ ± h·v float32'de θ'ya yuvarlanır -> fark ≈ 0, göreli hata ≈ 1
        self.assertGreater(self.err[("float32", 1e-10)], 0.1)


if __name__ == "__main__":
    unittest.main(verbosity=2)


In [ ]:
# === Ortam kontrolü ===
import torch, numpy, scipy, pandas, matplotlib
print('PyTorch', torch.__version__, '| NumPy', numpy.__version__, '| SciPy', scipy.__version__,
      '| pandas', pandas.__version__, '| Matplotlib', matplotlib.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK (CPU ile çalışılacak)')

## 1. Modeller
`ShallowCNN`: 3 konvolüsyon + 2 tam bağlantılı katman. `DeepCNN`: 8 konvolüsyon + 2 tam bağlantılı katman, **BatchNorm yok** (gradyan sönmesi/patlamasını gözlemlemek için).

In [ ]:
!python models.py

## 2. Test süreçleri (birim testleri)
Optimizer formüllerinin elle hesaplanan değerlerle ve PyTorch referansıyla eşleşmesi, determinizm, NaN/Inf tespiti, gradyan normu ölçümü ve sonlu fark doğrulaması test edilir (~1 dk).

In [ ]:
# === Rapor için doğrulama çıktıları: tests.py + gradient_check.py (~2-3 dk) ===
# Tek başına çalışır (proje klasöründe olmanız yeterli). Deney 1-2'yi YENİDEN ÇALIŞTIRMAZ.
import os, sys, subprocess
import numpy as np, pandas as pd
from IPython.display import Image, display

need = ['models.py', 'optimizers.py', 'gradient_check.py', 'tests.py', 'experiments.py']
assert all(os.path.exists(f) for f in need), 'Proje klasöründe değilsiniz: önce %cd ile geçin.'
os.makedirs('results', exist_ok=True); os.makedirs('plots', exist_ok=True)

# 1) Birim testleri -> results/tests_output.txt (raporun ekine konacak çıktı)
r = subprocess.run([sys.executable, '-m', 'unittest', 'tests', '-v'], capture_output=True, text=True)
log = r.stderr + r.stdout
with open('results/tests_output.txt', 'w', encoding='utf-8') as f:
    f.write(log)
print(log)
print('TÜM TESTLER GEÇTİ' if r.returncode == 0 else '!!! BAŞARISIZ TEST VAR – yukarıdaki çıktıya bakın')

# 2) Deney 3: otomatik türev vs merkezi sonlu fark (CPU, float64 / float32)
if not os.path.exists('results/exp3_fd_error.csv'):
    !python experiments.py --exp 3 --no-plot
fd = pd.read_csv('results/exp3_fd_error.csv')

# En küçük göreli hata ve onu veren h; teorik h* ~ eps_mach^(1/3) ile karşılaştırma
eps = {'float64': np.finfo(np.float64).eps, 'float32': np.finfo(np.float32).eps}
best = fd.loc[fd.groupby(['model', 'dtype']).rel_error.idxmin(),
              ['model', 'dtype', 'h', 'rel_error']].reset_index(drop=True)
best['teorik_h*'] = best.dtype.map(lambda d: eps[d] ** (1 / 3))
best.columns = ['Model', 'Hassasiyet', 'En iyi h', 'Min. göreli hata', 'Teorik h* ≈ ε^(1/3)']
best.to_csv('results/exp3_best_h.csv', index=False)
print('\nTablo – Sonlu fark doğrulaması (en iyi adım ve hata)')
display(best.style.format({'En iyi h': '{:.1e}', 'Min. göreli hata': '{:.2e}',
                           'Teorik h* ≈ ε^(1/3)': '{:.1e}'}))

# 3) Şekil 5: hata - h grafiği (log-log, V şekli)
import plot_results
plot_results.setup_style()
plot_results.plot_fd_error(fd, 'plots')
display(Image('plots/fd_error.png', width=900))

## 3. Canlı demo (~2 dk)
Küçük veri alt kümesiyle (6000 eğitim / 2000 test örneği, 2 epoch, 2 seed) tüm hattın çalıştığını gösterir. Sonuçlar `results_demo/` ve `plots_demo/` klasörlerine yazılır, asıl sonuçlara dokunmaz.

In [ ]:
!python experiments.py --quick --exp 1 2 --results-dir results_demo --plots-dir plots_demo

from IPython.display import Image, display
display(Image('plots_demo/loss_convergence.png', width=850))

## 4. Tam deneyler
Deney 1 (2 model × 4 optimizer × 5 seed × 10 epoch, ~50 dk) ve Deney 2 (DeepCNN, 4 optimizer × 5 öğrenme oranı × 3 seed, ~10 dk). `RUN_FULL = False` iken yalnızca sonucu eksik olan deney çalıştırılır; Drive'daki kayıtlı sonuçlar yeniden kullanılır.

In [ ]:
import pandas as pd

def eksik(*dosyalar):
    return not all(os.path.exists(f'results/{f}') for f in dosyalar)

calistir = []
if RUN_FULL or eksik('exp1_runs.csv', 'exp1_epochs.csv', 'exp1_layer_grads.csv'):
    calistir.append('1')
# Deney 2 artık 3 seed ile yapılıyor: eski tek seed'li sonuç varsa yeniden çalıştır
if RUN_FULL or eksik('exp2_lr_stability.csv') or \
        pd.read_csv('results/exp2_lr_stability.csv').seed.nunique() < 3:
    calistir.append('2')

if calistir:
    print('Çalıştırılacak deney(ler):', calistir)
    !python experiments.py --exp {' '.join(calistir)} --no-plot
if '1' not in calistir:
    print('Kayıtlı Deney 1 sonuçları kullanılıyor.')
    !python experiments.py --resummarize

## 5. Gradyan doğrulaması ve CPU–GPU karşılaştırması (~1-2 dk)
Deney 3: merkezi sonlu fark $D_h = \frac{L(\theta + h v) - L(\theta - h v)}{2h}$ ile otomatik türev $\nabla L \cdot v$ karşılaştırılır. Toplam hata $\approx \frac{h^2}{6}|L'''| + \frac{\varepsilon_{mach}|L|}{h}$; en iyi adım $h^* \sim \varepsilon_{mach}^{1/3}$.

Deney 4: iterasyon başına süre (ileri + geri yayılım + Adam adımı).

In [ ]:
if RUN_FULL or not all(os.path.exists(f'results/{f}') for f in ['exp3_fd_error.csv', 'exp4_cpu_gpu.csv']):
    !python experiments.py --exp 3 4 --no-plot
else:
    print('Kayıtlı Deney 3-4 sonuçları kullanılıyor.')

## 6. Hessian spektrumu ve kararlılık sınırı – kuvvet yöntemi (~3 dk)
Hessian $H=\nabla^2 L(\theta)$ açıkça kurulmaz ($P^2 \approx 10^{10}$ eleman); Hessian–vektör çarpımı $Hv = \nabla_\theta(\nabla_\theta L \cdot v)$ iki geri yayılımla hesaplanır. Kuvvet yöntemi: $w_k = Hv_{k-1}$, $\lambda_k = v_{k-1}^\top w_k$ (Rayleigh bölümü), $v_k = w_k/\|w_k\|$; hata $O(|\lambda_2/\lambda_1|^{2k})$.

Yerel ikinci derece modelde gradyan inişinin hatası $e_{k+1}=(I-\eta H)e_k$ olduğundan yöntem ancak $|1-\eta\lambda_i|<1$, yani $\eta < 2/\lambda_{max}$ iken kararlıdır. Bu teorik sınır Deney 2'deki deneysel sınırla karşılaştırılır.

In [ ]:
if RUN_FULL or not os.path.exists('results/exp5_sharpness.csv'):
    !python experiments.py --exp 5 --no-plot
else:
    print('Kayıtlı Deney 5 sonuçları kullanılıyor.')
display(pd.read_csv('results/exp5_sharpness.csv').groupby(['model', 'epoch'])[['lambda_max', 'eta_crit_gd', 'power_iters']]
        .agg(['mean', 'std']).round(4))

## 7. Ablasyon (~10 dk)
(a) **BatchNorm:** DeepCNN'de SGD'nin takılmasının nedeni gerçekten gradyan sönmesi ise, her konvolüsyondan sonra BatchNorm eklenince SGD öğrenmelidir. (b) **Adam yanlılık düzeltmesi:** $m_0=v_0=0$ başlangıcı nedeniyle $E[m_t]=(1-\beta_1^t)E[g]$; düzeltme kapatılınca ilk adım $\eta(1-\beta_1)/\sqrt{1-\beta_2}\approx 3.16\eta$ olur. Deney 1 ile aynı 5 seed kullanıldığı için karşılaştırma eşleştirilmiş t-testiyle yapılır.

In [ ]:
if RUN_FULL or not os.path.exists('results/exp6_runs.csv'):
    !python experiments.py --exp 6 --no-plot
else:
    print('Kayıtlı Deney 6 sonuçları kullanılıyor.')
display(pd.read_csv('results/exp6_summary.csv'))

## 8. Sonuç tabloları

In [ ]:
import pandas as pd
pd.set_option('display.width', 250, 'display.max_columns', 30)

tablolar = [
    ('Tablo 1 – Deney 1: Yakınsama ve seed kararlılığı (ortalama ± std, 5 seed)', 'exp1_summary.csv'),
    ('Tablo 2 – Deney 2: Öğrenme oranı kararlılığı (DeepCNN, 3 seed özeti)', 'exp2_summary.csv'),
    ('Tablo 3 – Deney 3: Sonlu fark doğrulaması (en iyi h)', 'exp3_best_h.csv'),
    ('Tablo 4 – Deney 4: CPU / GPU', 'exp4_cpu_gpu.csv'),
    ('Tablo 5 – Deney 6: Ablasyon (eşleştirilmiş t-testi)', 'exp6_summary.csv'),
]
for baslik, dosya in tablolar:
    if os.path.exists(f'results/{dosya}'):
        print(baslik)
        display(pd.read_csv(f'results/{dosya}'))

## 9. Grafikler (300 DPI, `plots/`)

In [ ]:
!python plot_results.py

from IPython.display import Image, Markdown, display
captions = {
    'loss_convergence': 'Şekil 1 – Eğitim kaybı (log ölçek), 5 seed ortalaması ± std',
    'gradient_norm':    'Şekil 2 – Katman bazlı gradyan normları ve ilk/son katman oranı',
    'lr_heatmap':       'Şekil 3 – Öğrenme oranı – optimizer kararlılık haritası (DeepCNN)',
    'time_vs_acc':      'Şekil 4 – Eğitim süresi – test doğruluğu',
    'fd_error':         'Şekil 5 – Sonlu fark göreli hatası (kesme / yuvarlama dengesi)',
    'cpu_gpu':          'Şekil 6 – CPU / GPU iterasyon süresi',
    'sharpness':        'Şekil 7 – Kuvvet yöntemi yakınsaması, λ_max ve teorik kararlılık sınırı 2/λ_max',
    'ablation':         'Şekil 8 – Ablasyon: BatchNorm ve Adam yanlılık düzeltmesi',
}
for name, cap in captions.items():
    path = f'plots/{name}.png'
    if os.path.exists(path):
        display(Markdown(f'**{cap}**'))
        display(Image(path, width=900))

## 10. Çıktıları indir
results/, plots/ ve logs/ klasörlerini (ortam bilgisi, defterdeki tüm konsol çıktıları, çalıştırılmış defter) tek bir `MAT353_Ciktilar.zip` arşivinde toplar ve bilgisayara indirir. Tüm hücreler bittikten sonra çalıştırın.

In [ ]:
# === Tüm çıktıları tek ZIP olarak indir: MAT353_Ciktilar.zip ===
# results/ + plots/ + logs/ (ortam bilgisi, defterdeki tüm konsol çıktıları, çalıştırılmış defter)
import os, re, json, zipfile, platform, datetime

if 'PROJ' in globals():
    os.chdir(PROJ)
os.makedirs('logs', exist_ok=True)

# 1) Ortam bilgisi -> logs/ortam.txt
import torch, numpy, scipy, pandas, matplotlib
with open('logs/ortam.txt', 'w', encoding='utf-8') as f:
    f.write(f'Tarih     : {datetime.datetime.now():%Y-%m-%d %H:%M}\n'
            f'Python    : {platform.python_version()}\n'
            f'PyTorch   : {torch.__version__} (CUDA {torch.version.cuda})\n'
            f'NumPy {numpy.__version__} | SciPy {scipy.__version__} | pandas {pandas.__version__} '
            f'| Matplotlib {matplotlib.__version__}\n'
            f'GPU       : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK"}\n'
            f'CPU       : {platform.processor() or platform.machine()} ({os.cpu_count()} çekirdek)\n')

# 2) Defterdeki tüm hücre çıktıları -> logs/notebook_outputs.txt (+ çalıştırılmış .ipynb)
ansi = re.compile(r'\x1b\[[0-9;]*[A-Za-z]')
try:
    from google.colab import _message
    nb = _message.blocking_request('get_ipynb', request='', timeout_sec=120)['ipynb']
    parcalar = []
    for i, c in enumerate(nb['cells']):
        src = ''.join(c.get('source', ''))
        if c['cell_type'] != 'code' or src.startswith('%%writefile'):
            continue
        metinler = []
        for o in c.get('outputs', []):
            if o['output_type'] == 'stream':
                metinler.append(''.join(o['text']))
            elif o['output_type'] == 'error':
                metinler.append('\n'.join(o.get('traceback', [])))
            elif 'text/plain' in o.get('data', {}):
                t = ''.join(o['data']['text/plain'])
                if not t.startswith('<IPython'):
                    metinler.append(t)
        if metinler:
            baslik = src.strip().splitlines()[0] if src.strip() else ''
            parcalar.append(f'{"=" * 70}\nHücre {i}: {baslik}\n{"=" * 70}\n' + ansi.sub('', '\n'.join(metinler)))
    with open('logs/notebook_outputs.txt', 'w', encoding='utf-8') as f:
        f.write('\n\n'.join(parcalar))
    with open('logs/MAT353_Proje_calistirilmis.ipynb', 'w', encoding='utf-8') as f:
        json.dump(nb, f, ensure_ascii=False, indent=1)
    print(f'Konsol çıktıları kaydedildi: {len(parcalar)} hücre -> logs/notebook_outputs.txt')
except Exception as e:
    print('Defter çıktıları alınamadı (yalnızca Colab\'da çalışır):', e)

# 3) Beklenen dosyaların kontrolü
beklenen = {
    'results': ['exp1_summary.csv', 'exp1_epochs.csv', 'exp1_layer_grads.csv', 'exp1_runs.csv',
                'exp2_lr_stability.csv', 'exp2_summary.csv', 'exp2_epochs.csv',
                'exp3_fd_error.csv', 'exp3_best_h.csv', 'exp4_cpu_gpu.csv',
                'exp5_sharpness.csv', 'exp5_power_history.csv',
                'exp6_summary.csv', 'exp6_runs.csv', 'exp6_epochs.csv', 'tests_output.txt'],
    'plots': ['loss_convergence.png', 'gradient_norm.png', 'lr_heatmap.png', 'time_vs_acc.png',
              'fd_error.png', 'cpu_gpu.png', 'sharpness.png', 'ablation.png'],
}
eksikler = [f'{d}/{f}' for d, fs in beklenen.items() for f in fs if not os.path.exists(f'{d}/{f}')]
print('Tüm beklenen dosyalar mevcut.' if not eksikler else f'!!! EKSİK {len(eksikler)} dosya: {eksikler}')

# 4) ZIP (Drive senkron gecikmesi olmasın diye /content altına yazılır) ve indirme
zip_yolu = '/content/MAT353_Ciktilar.zip' if os.path.isdir('/content') else 'MAT353_Ciktilar.zip'
n = 0
with zipfile.ZipFile(zip_yolu, 'w', zipfile.ZIP_DEFLATED) as z:
    for klasor in ['results', 'plots', 'logs']:
        for kok, _, dosyalar in os.walk(klasor):
            for ad in sorted(dosyalar):
                z.write(os.path.join(kok, ad), arcname=os.path.join('MAT353_Ciktilar', kok, ad))
                n += 1
print(f'{zip_yolu}: {n} dosya, {os.path.getsize(zip_yolu) / 1e6:.1f} MB')

try:
    from google.colab import files
    files.download(zip_yolu)
except ImportError:
    print('Colab dışında: ZIP dosyası', os.path.abspath(zip_yolu))
